# 09. Portfolio Performance & Sensitivity Analysis

Una vez calculadas las rentabilidades netas tras descontar el impacto de la fricción operativa y el turnover en el notebook anterior, se procede a la evaluación económica final de las carteras. Este bloque tiene como objetivo analizar en profundidad el rendimiento ajustado al riesgo de las distintas estrategias, evaluar la estabilidad temporal de las métricas en diferentes subperiodos OOS, estudiar la sensibilidad de los resultados ante variaciones en los parámetros clave de ejecución y establecer la comparativa definitiva entre los modelos de Machine Learning y los benchmarks de referencia.

## 1. Imports & Configuration

### 1.1 Librerías

In [1]:
import sys
import pandas as pd 
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path
from scipy.stats import spearmanr

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

### 1.2 Configuración del notebook

In [2]:
# =============================================================================
# Experiment Configuration
# =============================================================================

PREDICTION_HORIZON = 21

TRADING_DAYS_PER_YEAR = 252

RISK_FREE_RATE = 0.0

REBALANCING_FREQUENCY = 21

BASE_TRANSACTION_COST = 0.0015 


### 1.3 Rutas y parámetros globales

In [3]:
# =============================================================================
# Paths
# =============================================================================

PRICES_PATH = "../data/raw/sp500_prices_extended.parquet"

RAW_WEIGHTS_PATH = "../data/portfolio_results/final_portfolio_weights.parquet"

NET_RETURNS = "../data/portfolio_results/net_portfolio_returns.parquet"

## 2. Data Loading

In [4]:
# =============================================================================
# Load Extended Price Data
# =============================================================================

prices = pd.read_parquet("../data/raw/sp500_prices_extended.parquet")

# =============================================================================
# Compute Daily Asset Returns
# =============================================================================

adj_close = prices["Adj Close"]

asset_returns = adj_close.pct_change()

# =============================================================================
# Load Net Portfolio Returns
# =============================================================================

net_returns = pd.read_parquet(
    "../data/portfolio_results/net_portfolio_returns.parquet"
)


# =============================================================================
# Load Final Portfolio Weights
# =============================================================================

weights_raw = pd.read_parquet(
    "../data/portfolio_results/final_portfolio_weights.parquet"
)

## 3. Portfolio Performance & Sensitivity Analysis

Una vez obtenidas las rentabilidades netas de las distintas estrategias, se realiza la evaluación económica final de las carteras. Este bloque tiene tres objetivos: comparar el rendimiento ajustado al riesgo de las estrategias, estudiar la sensibilidad de los resultados ante cambios en la frecuencia de rebalanceo y establecer una comparación final entre los modelos de Machine Learning y las estrategias de referencia.


## 3. Global Net Performance Evaluation

Tras someter las estrategias a las fricciones de ejecución en el último capítulo del notebook 08, la evaluación final de rendimiento debe realizarse de forma estricta sobre las **rentabilidades netas** de comisiones. Medir el valor generado únicamente mediante la rentabilidad acumulada resulta insuficiente en la gestión cuantitativa moderna: es imprescindible contrastar el retorno obtenido contra la volatilidad asumida, la severidad de las caídas patrimoniales y la velocidad de recuperación del capital.

Para llevar a cabo un diagnóstico exhaustivo e imparcial, este apartado articula el análisis *out-of-sample* (OOS) en torno a tres dimensiones analíticas complementarias:

* **Métricas de rentabilidad y volatilidad:** Miden la magnitud del crecimiento patrimonial y el nivel de dispersión diario.
    * **Cumulative Return:** Rentabilidad acumulada total a lo largo del horizonte *out-of-sample*.
    * **CAGR (*Compound Annual Growth Rate*):** Tasa de crecimiento anual compuesta que normaliza el rendimiento temporal.
    * **Annualized Volatility:** Volatilidad de los retornos diarios multiplicada por el factor de anualización ($\sqrt{252}$).


* **Métricas de rendimiento ajustado al riesgo:** Evalúan la eficiencia en la conversión de riesgo en retorno.
    * **Sharpe Ratio:** Exceso de rentabilidad por unidad de volatilidad total.
    * **Sortino Ratio:** Exceso de retorno por unidad de volatilidad a la baja (*downside risk*), penalizando únicamente los retornos negativos.
    * **Calmar Ratio:** Relación entre el CAGR y el *Maximum Drawdown*, que mide el retorno obtenido por cada unidad de pérdida máxima histórica.


* **Métricas de *drawdown* y comportamiento en momentos de estrés:** Analizan la resiliencia del capital ante fases adversas de mercado.
    * **Maximum Drawdown (MDD):** Mayor pérdida porcentual acumulada desde un máximo histórico previo (*peak-to-trough*).
    * **Average Drawdown:** Pérdida media observada a lo largo de todos los episodios de *drawdown*, evitando que la evaluación de riesgo dependa exclusivamente de un único evento extremo puntual.
    * **Underwater Duration (o Duration in Drawdown)**: Número máximo de sesiones de negociación consecutivas en las que el capital se encuentra por debajo de su máximo histórico anterior.

Este marco multidimensional permite determinar qué arquitecturas de predicción y esquemas de ponderación no solo generan alfa bruto, sino que logran consolidar carteras sólidas, eficientes e inmunes a la erosión operativa en el entorno real.


### 3.1 Overall Net Performance

In [5]:
from src.portfolio.utils import calculate_drawdown_metrics

# =============================================================================
# Comparative Performance Analysis
# =============================================================================

TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0


net_returns["date"] = pd.to_datetime(
    net_returns["date"]
)

net_returns = (
    net_returns
    .sort_values(
        ["model", "portfolio", "date"]
    )
    .reset_index(drop=True)
)

# =============================================================================
# Performance Calculation
# =============================================================================

performance_results = []


for (
    model,
    portfolio,
), group in (
    net_returns
    .groupby(
        [
            "model",
            "portfolio",
        ]
    )
):

    group = (
        group
        .sort_values("date")
        .copy()
    )

    returns = (
        group["net_return_base"]
        .dropna()
    )

    if len(returns) == 0:
        continue

    # -------------------------------------------------------------------------
    # Time Horizon
    # -------------------------------------------------------------------------

    observations = len(returns)

    years = (
        observations
        / TRADING_DAYS_PER_YEAR
    )

    # -------------------------------------------------------------------------
    # Cumulative Return
    # -------------------------------------------------------------------------

    cumulative_return = (
        (1.0 + returns).prod()
        - 1.0
    )

    # -------------------------------------------------------------------------
    # CAGR
    # -------------------------------------------------------------------------

    if years > 0:

        cagr = (
            (1.0 + cumulative_return)
            ** (1.0 / years)
            - 1.0
        )

    else:

        cagr = np.nan

    # -------------------------------------------------------------------------
    # Annualized Volatility
    # -------------------------------------------------------------------------

    daily_volatility = (
        returns.std(
            ddof=1
        )
    )

    annualized_volatility = (
        daily_volatility
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    # -------------------------------------------------------------------------
    # Sharpe Ratio
    # -------------------------------------------------------------------------

    daily_rf = (
        RISK_FREE_RATE
        / TRADING_DAYS_PER_YEAR
    )

    excess_returns = (
        returns
        - daily_rf
    )

    if daily_volatility > 0:

        sharpe_ratio = (
            excess_returns.mean()
            / daily_volatility
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

    else:

        sharpe_ratio = np.nan

    # -------------------------------------------------------------------------
    # Sortino Ratio
    # -------------------------------------------------------------------------

    downside_diff = (
        returns
        - daily_rf
    )

    downside_returns = (
        downside_diff[
            downside_diff < 0
        ]
    )

    if len(downside_returns) > 0:

        # Denominator uses total number of observations
        downside_deviation = (
            np.sqrt(
                np.sum(
                    downside_returns ** 2
                )
                / len(returns)
            )
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

    else:

        downside_deviation = np.nan

    if (
        pd.notna(
            downside_deviation
        )
        and downside_deviation > 0
    ):

        sortino_ratio = (
            (
                cagr
                - RISK_FREE_RATE
            )
            / downside_deviation
        )

    else:

        sortino_ratio = np.nan

    # -------------------------------------------------------------------------
    # Drawdown Metrics
    # -------------------------------------------------------------------------

    (
        maximum_drawdown,
        average_drawdown,
        maximum_underwater_duration,
    ) = calculate_drawdown_metrics(
        returns
    )

    # -------------------------------------------------------------------------
    # Calmar Ratio
    # -------------------------------------------------------------------------

    if maximum_drawdown < 0:

        calmar_ratio = (
            cagr
            / abs(
                maximum_drawdown
            )
        )

    else:

        calmar_ratio = np.nan

    # -------------------------------------------------------------------------
    # Store Results
    # -------------------------------------------------------------------------

    performance_results.append(
        {
            "model": model,
            "portfolio": portfolio,
            "observations": observations,
            "cumulative_return": cumulative_return,
            "CAGR": cagr,
            "annualized_volatility":
                annualized_volatility,
            "Sharpe": sharpe_ratio,
            "Sortino": sortino_ratio,
            "Calmar": calmar_ratio,
            "maximum_drawdown":
                maximum_drawdown,
            "average_drawdown":
                average_drawdown,
            "maximum_underwater_duration_days":
                maximum_underwater_duration,
        }
    )


# =============================================================================
# Consolidate Performance Metrics
# =============================================================================

performance_metrics = (
    pd.DataFrame(
        performance_results
    )
    .sort_values(
        "Sharpe",
        ascending=False,
    )
    .reset_index(drop=True)
)


# =============================================================================
# Validation
# =============================================================================

assert (
    performance_metrics[
        "model"
    ].nunique()
    == net_returns[
        "model"
    ].nunique()
)

assert (
    performance_metrics[
        "portfolio"
    ].nunique()
    == net_returns[
        "portfolio"
    ].nunique()
)

assert (
    performance_metrics[
        [
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "maximum_drawdown",
        ]
    ]
    .notna()
    .all()
    .all()
)


# =============================================================================
# Audit Output
# =============================================================================

print("=" * 80)
print("3.1 — NET PORTFOLIO PERFORMANCE")
print("=" * 80)

print(
    f"✓ Strategies evaluated = "
    f"{len(performance_metrics):,}"
)

print(
    f"✓ Models evaluated = "
    f"{performance_metrics['model'].nunique():,}"
)

print(
    f"✓ Date range = "
    f"{net_returns['date'].min().date()} "
    f"→ "
    f"{net_returns['date'].max().date()}"
)

print(
    f"✓ Missing performance metrics = "
    f"{performance_metrics.isna().sum().sum():,}"
)

print()

print(
    performance_metrics[
        [
            "model",
            "portfolio",
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
            "maximum_underwater_duration_days",
        ]
    ]
    .to_string(
        index=False,
        formatters={
            "CAGR":
                lambda x: f"{x:.2%}",
            "annualized_volatility":
                lambda x: f"{x:.2%}",
            "Sharpe":
                lambda x: f"{x:.3f}",
            "Sortino":
                lambda x: f"{x:.3f}",
            "Calmar":
                lambda x: f"{x:.3f}",
            "maximum_drawdown":
                lambda x: f"{x:.2%}",
        },
    )
)

3.1 — NET PORTFOLIO PERFORMANCE
✓ Strategies evaluated = 42
✓ Models evaluated = 3
✓ Date range = 2025-01-16 → 2026-08-10
✓ Missing performance metrics = 0

        model                           portfolio   CAGR annualized_volatility Sharpe Sortino Calmar maximum_drawdown  maximum_underwater_duration_days
        Ridge      long_only_top_10_signal_weight 50.90%                26.25%  1.699   2.937  1.889          -26.94%                               108
        Ridge              long_only_equal_weight 47.60%                25.45%  1.658   2.850  1.793          -26.54%                               109
        Ridge      long_only_top_20_signal_weight 45.69%                25.13%  1.624   2.777  1.719          -26.58%                               108
      XGBoost      long_only_top_20_signal_weight 55.79%                30.81%  1.593   2.716  1.913          -29.16%                                81
Random Forest      long_only_top_10_signal_weight 67.01%                36.73%  1.5

El análisis comparativo de las estrategias evaluadas revela diferencias sustanciales en el rendimiento ajustado por riesgo y la magnitud del drawdown, en función del modelo de aprendizaje automático y del esquema de construcción de cartera utilizados.

En términos de eficiencia ajustada por riesgo, la combinación de Ridge con la cartera long_only_top_10_signal_weight alcanza el Sharpe Ratio más elevado de la muestra ($1.699$), secundado por una cartera con ponderación equitativa (long_only_equal_weight, $1.658$) sobre el mismo modelo. Esta configuración de Ridge también registra la mejor asimetría en la rentabilidad ajustada a la baja, con un Sortino Ratio de $2.937$.

Desde la perspectiva de la rentabilidad absoluta, los modelos no lineales lideran la comparativa: Random Forest junto con long_only_top_10_signal_weight genera el mayor CAGR del estudio ($67.01\%$), seguido de la variante equivalente en XGBoost ($62.15\%$). No obstante, esta mayor rentabilidad viene acompañada de una mayor volatilidad anualizada ($36.73\%$ y $35.60\%$, respectivamente) y de caídas máximas (maximum drawdown) más pronunciadas, que superan el $-32\%$.

El control del riesgo destaca en las estrategias Long-Short y en las configuraciones de Paridad de Riesgo. Las carteras long_short_equal_weight exhiben los drawdowns máximos más acotados de la muestra ($-12.24\%$ para Ridge y $-13.22\%$ para XGBoost), manteniendo volatilidades anualizadas contenidas entre el $11.81\%$ y el $15.07\%$. Por su parte, la combinación de Random Forest con long_only_top_20_risk_parity logra el periodo de permanencia bajo máximos (maximum underwater duration) más breve, con solo $57$ días de negociación.

En el extremo opuesto, las carteras optimizadas mediante Maximum Sharpe y las estrategias basadas en Inverse Volatility de $30$ activos se sitúan de forma consistente en el tramo inferior de la tabla, con ratios de Sharpe ajustados entre $1.009$ y $1.357$, e incrementos significativos en la duración de las etapas de pérdida.

### 3.2 Agregated Performance Analysis

Para sintetizar los resultados, las métricas se agrupan promediando el rendimiento de las carteras en tres dimensiones analíticas: el **modelo** de aprendizaje automático subyacente (*Ridge*, *XGBoost* y *Random Forest*), la **filosofía** de construcción de cartera utilizada, y el **tamaño del universo** de activos seleccionados (*Top 10*, *Top 20* y *Top 30*).

In [6]:
# =============================================================================
# Performance Analysis by Model, Portfolio Type and Universe
# =============================================================================

# Columns available in performance_metrics:
# model
# portfolio
# CAGR
# annualized_volatility
# Sharpe
# Sortino
# Calmar
# maximum_drawdown
# maximum_underwater_duration_days


# =============================================================================
# 1. Performance by ML Model
# =============================================================================

model_comparison = (
    performance_metrics
    .groupby("model")[
        [
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
            "maximum_underwater_duration_days",
        ]
    ]
    .mean()
    .sort_values("Sharpe", ascending=False)
)

print("=" * 80)
print("3.2 — PERFORMANCE BY ML MODEL")
print("=" * 80)

print(
    model_comparison.to_string(
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
            "maximum_underwater_duration_days": lambda x: f"{x:.0f}",
        }
    )
)


# =============================================================================
# 2. Performance by Portfolio Philosophy
# =============================================================================

def extract_weighting_scheme(portfolio):
    """
    Extract the portfolio weighting philosophy from the portfolio name.
    """
    portfolio = portfolio.lower()

    if "equal_weight" in portfolio:
        return "Equal Weight"

    if "inverse_volatility" in portfolio:
        return "Inverse Volatility"

    if "risk_parity" in portfolio:
        return "Risk Parity"

    if "signal_weighting" in portfolio:
        return "Signal Weighting"

    if "maximum_sharpe" in portfolio:
        return "Maximum Sharpe"

    return "Other"


performance_metrics["weighting_scheme"] = (
    performance_metrics["portfolio"]
    .apply(extract_weighting_scheme)
)

portfolio_type_comparison = (
    performance_metrics
    .groupby("weighting_scheme")[
        [
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
            "maximum_underwater_duration_days",
        ]
    ]
    .mean()
    .sort_values("Sharpe", ascending=False)
)

print("\n" + "=" * 80)
print("3.2 — PERFORMANCE BY PORTFOLIO PHILOSOPHY")
print("=" * 80)

print(
    portfolio_type_comparison.to_string(
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
            "maximum_underwater_duration_days": lambda x: f"{x:.0f}",
        }
    )
)


# =============================================================================
# 3. Performance by Portfolio Universe
# =============================================================================

def extract_universe(portfolio):
    """
    Extract the investment universe from the portfolio name.
    """
    portfolio = portfolio.lower()

    if "top_10" in portfolio:
        return "Top 10%"

    if "top_20" in portfolio:
        return "Top 20%"

    if "top_30" in portfolio:
        return "Top 30%"

    # Baseline portfolios without quantile selection
    return "Full / Baseline"


performance_metrics["universe"] = (
    performance_metrics["portfolio"]
    .apply(extract_universe)
)

universe_comparison = (
    performance_metrics
    .groupby("universe")[
        [
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
            "maximum_underwater_duration_days",
        ]
    ]
    .mean()
)


universe_order = [
    "Full / Baseline",
    "Top 10%",
    "Top 20%",
    "Top 30%",
]

universe_comparison = (
    universe_comparison
    .reindex(
        [x for x in universe_order if x in universe_comparison.index]
    )
)

print("\n" + "=" * 80)
print("3.2 — PERFORMANCE BY PORTFOLIO UNIVERSE")
print("=" * 80)

print(
    universe_comparison.to_string(
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
            "maximum_underwater_duration_days": lambda x: f"{x:.0f}",
        }
    )
)

3.2 — PERFORMANCE BY ML MODEL
                CAGR annualized_volatility Sharpe Sortino Calmar maximum_drawdown maximum_underwater_duration_days
model                                                                                                             
XGBoost       37.08%                23.65%  1.407   2.287  1.600          -22.63%                               88
Random Forest 36.38%                23.93%  1.368   2.212  1.596          -22.27%                               83
Ridge         31.21%                21.63%  1.328   2.127  1.364          -22.47%                              106

3.2 — PERFORMANCE BY PORTFOLIO PHILOSOPHY
                     CAGR annualized_volatility Sharpe Sortino Calmar maximum_drawdown maximum_underwater_duration_days
weighting_scheme                                                                                                       
Other              52.12%                29.45%  1.571   2.660  1.827          -28.41%                          

Al agregar los datos por modelo, XGBoost lidera en retorno anualizado ($37.08\%$) y en ratios de eficiencia (Sharpe de $1.407$ y Sortino de $2.287$), seguido muy de cerca por Random Forest. Por su parte, Ridge presenta un comportamiento promedio con menor volatilidad ($21.63\%$), aunque su tiempo de recuperación medio se extiende a $106$ días.

Respecto a la filosofía de ponderación, la categoría Other (que engloba los esquemas basados en intensidad de señal y Long-Short) destaca con el CAGR más elevado ($52.12\%$) y el mayor Sharpe ($1.571$). Entre las estrategias estándar, Equal Weight ofrece el mejor equilibrio ajustado por riesgo ($1.488$), mientras que Maximum Sharpe y Risk Parity contienen la volatilidad por debajo del $21\%$, a costa de una menor rentabilidad global.

Por último, la dimensión del universo de activos muestra una clara relación entre concentración y rendimiento: seleccionar el Top 10% maximiza la rentabilidad ($44.63\%$), pero incrementa la volatilidad ($27.65\%$) y el drawdown máximo ($-27.20\%$). A medida que el universo se amplía hacia el Top 20% y Top 30%, el perfil de riesgo se modera de forma progresiva, reduciendo tanto la volatilidad como la rentabilidad final.

## 4. Performance Interpretation

El comportamiento divergente observado entre los modelos, las metodologías de construcción de cartera y el tamaño de los universos responde a mecánicas económico-financieras y cuantitativas bien documentadas.

### 4.1 Efecto de Regularización vs. Captura de No Linealidades

La superioridad de *Ridge* en los ratios de eficiencia (*Sharpe* y *Sortino* en cartera individual) radica en su penalización L2, la cual reduce la varianza de los coeficientes frente a la colinealidad de los *factors* de mercado. Al emitir estimaciones de retorno más conservadoras y estables, minimiza las rotaciones innecesarias y el ruido de asignación. Por el contrario, *XGBoost* y *Random Forest* destacan en retorno absoluto al mapear interacciones no lineales complejas entre *features*; sin embargo, esta agresividad predictiva induce una mayor volatilidad estructural en las ponderaciones, aumentando las caídas temporales.

### 4.2 El Ratio de Información de la Señal (Signal Weighting)

El elevado desempeño del esquema *Signal Weight* confirma la existencia de un alpha monótono en las predicciones: la magnitud del valor predicho por los modelos no solo indica la dirección del activo, sino también la convicción de la anomalía. Ponderar en función de la intensidad de la señal maximiza la captura de este alpha en comparación con asignaciones pasivas (*Equal Weight*), compensando el incremento asumido en la volatilidad.

### 4.3 Fallo out-of-sample de la Optimización Media-Varianza

El bajo rendimiento relativo de las carteras *Maximum Sharpe* ilustra el dilema clásico de la optimización markowitziana: la maximización del ratio dentro de la muestra convierte los errores de estimación en las matrices de covarianza y vectores de retorno esperado en ponderaciones extremas. Al evaluar estas posiciones *out-of-sample*, la inestabilidad de los pesos penaliza severamente el rendimiento ajustado por riesgo.

### 4.4 Trade-off de Concentración y Diversificación

La degradación progresiva del *CAGR* al pasar del *Top 10%* al *Top 30%* evidencia que la capacidad predictiva del modelo se concentra en los extremos de la distribución (*tail alpha*). Incorporar un mayor número de activos diluye la prima de riesgo capturada por la señal ML a cambio de reducir la volatilidad total, confirmando que el poder discriminatorio del modelo decrece rápidamente a medida que se desciende en el ranking de predicción.

### 4.5 Simetría y Cobertura en Carteras Long-Short

La contención en las métricas de *drawdown* de las carteras *Long-Short* responde a la neutralización parcial del riesgo sistemático (*Beta*). Al tomar posiciones cortas en los activos con peores señales predichas, la estrategia aísla el rendimiento idiosincrásico de la selección de activos, sacrificando la prima de riesgo del mercado a cambio de una trayectoria de capital sustancialmente más estable durante episodios de estrés bursátil.

### 4.6 Rentabilidad Bruta frente a Rentabilidad Neta

La comparación entre gross performance y net performance constituye un filtro adicional de robustez. Una estrategia que presenta un elevado CAGR bruto pero requiere una rotación excesiva puede perder buena parte de su ventaja una vez incorporados los costes de transacción. Por tanto, la rentabilidad neta debe considerarse la métrica económicamente relevante para evaluar la viabilidad de implementación. Este efecto es especialmente importante en Maximum Sharpe, donde la elevada sensibilidad de los pesos genera una penalización operativa sustancial.

### 4.7 Rentabilidad Ajustada por Riesgo frente a Rentabilidad Absoluta

Finalmente, el CAGR no debe interpretarse de forma aislada. El Sharpe y el Sortino permiten evaluar cuánto retorno se obtiene por unidad de riesgo, mientras que el Calmar relaciona la rentabilidad anualizada con el Maximum Drawdown. La duración del período underwater añade una dimensión temporal que permite distinguir entre una caída profunda pero rápidamente recuperada y una estrategia que permanece durante largos períodos por debajo de sus máximos históricos. Por ello, la selección final de estrategias debe basarse en el conjunto de métricas y no exclusivamente en la rentabilidad acumulada.

## 5. Rebalancing Frequency Sensitivity Analysis


### 5.1 Sensitivity Framework & Frequency Grid

La selección de la frecuencia de rebalanceo representa uno de los compromisos (*trade-offs*) más críticos en la gestión de carteras cuantitativas. Mientras que una reponderación muy frecuente permite adaptar rápidamente las posiciones a la fuerza cambiante de las señales predictivas, también dispara la rotación de activos y la acumulación de costes de transacción. Por el contrario, frecuencias demasiado dilatadas reducen la fricción operativa a costa de permitir el desalineamiento (*drift*) respecto a las ponderaciones óptimas y diluir la capacidad del modelo para capturar ineficiencias de mercado.

Para identificar la ventana temporal que maximiza la eficiencia neta, este apartado evalúa la sensibilidad de los modelos ante cuatro horizontes de rebalanceo alternativos: **5 días** (semanal), **10 días** (bisemanal), **21 días** (mensual, utilizado como escenario base), **42 días** (bimensual) y **63 días** (trimestral).

La comparación entre horizontes se estructura analizando de forma simultánea el comportamiento operativo y el rendimiento final a través del siguiente bloque de métricas:

* **Dinámica operativa y fricción:** Se cuantifica la tasa de rotación anualizada (*Turnover*) y la erosión acumulada por costes de ejecución (*Transaction Costs*), permitiendo aislar el desgaste generado por cada frecuencia.
* **Preservación de retorno y riesgo:** Se evalúa la evolución de la rentabilidad bruta frente a la neta mediante el **CAGR neto** —establecido como la métrica central de decisión— y la rentabilidad acumulada (*Cumulative Return*).
* **Eficiencia y resiliencia:** Se mide la conservación del perfil de riesgo-retorno ajustado a través del *Sharpe Ratio*, *Sortino Ratio*, la profundidad de las caídas (*Maximum Drawdown*) y la velocidad de recuperación del capital (*Recovery Time*).

El objetivo de este análisis de sensibilidad es localizar el punto de equilibrio óptimo donde la frescura de la señal predictiva compense con creces la fricción de negociación, fundamentando empíricamente la elección del calendario de rebalanceo para su aplicación en entornos reales de inversión.

### 5.2 Frequency Sensitivity — Execution Engine

In [7]:
# =============================================================================
# Rebalancing Frequency Sensitivity
# =============================================================================

REBALANCING_FREQUENCIES = [5, 10, 21, 42, 63]

# Load final portfolio weights and extended S&P 500 asset prices
weights_raw = pd.read_parquet(
    "../data/portfolio_results/final_portfolio_weights.parquet"
)

prices = pd.read_parquet(
    "../data/raw/sp500_prices_extended.parquet"
)

weights_raw["date"] = pd.to_datetime(
    weights_raw["date"]
)

# =============================================================================
# Compute Daily Asset Returns
# =============================================================================

adj_close = prices["Adj Close"]

asset_returns = adj_close.pct_change()

asset_returns.index = pd.to_datetime(
    asset_returns.index
)

all_trading_dates = pd.Index(
    sorted(asset_returns.index.unique())
)

# =============================================================================
# Frequency Sensitivity — Execution Engine
# =============================================================================

frequency_results = []


for REBALANCING_FREQUENCY in REBALANCING_FREQUENCIES:

    # -------------------------------------------------------------------------
    # Construct & Filter Rebalancing Calendar
    # -------------------------------------------------------------------------

    all_weight_dates = (
        weights_raw["date"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    # Build rebalancing grid
    rebalancing_dates = (
        all_weight_dates.iloc[
            ::REBALANCING_FREQUENCY
        ]
    )

    # Filter weights dataset strictly to active rebalancing dates
    weights_df = weights_raw[
        weights_raw["date"].isin(
            rebalancing_dates
        )
    ].copy()

    # -------------------------------------------------------------------------
    # Rebalancing Calendar
    # -------------------------------------------------------------------------

    rebal_dates = sorted(
        weights_df["date"].unique()
    )

    # -------------------------------------------------------------------------
    # Validate Rebalancing Frequency Grid
    # -------------------------------------------------------------------------

    rebal_positions = [
        all_trading_dates.get_loc(date)
        for date in rebal_dates
    ]

    rebal_intervals = np.diff(
        rebal_positions
    )

    assert np.all(
        rebal_intervals
        == REBALANCING_FREQUENCY
    ), (
        f"Rebalancing dates do not follow "
        f"the expected "
        f"{REBALANCING_FREQUENCY}-day frequency."
    )

    # -------------------------------------------------------------------------
    # Execution Engine
    # -------------------------------------------------------------------------

    gross_returns_list = []

    grouped_weights = (
        weights_df
        .groupby(
            [
                "model",
                "portfolio",
            ]
        )
    )

    for (
        model,
        portfolio,
    ), group in grouped_weights:

        # ---------------------------------------------------------------------
        # Pivot target weight matrix
        # ---------------------------------------------------------------------

        weight_matrix = (
            group
            .pivot(
                index="date",
                columns="ticker",
                values="weight",
            )
            .fillna(0.0)
        )

        # ---------------------------------------------------------------------
        # Prevent look-ahead bias
        #
        # Weights generated at t_k become effective at t_k + 1.
        # Forward-fill maintains the portfolio allocation between
        # consecutive rebalancing dates.
        # ---------------------------------------------------------------------

        executed_weights = (
            weight_matrix
            .reindex(
                all_trading_dates
            )
            .shift(1)
            .ffill()
        )

        # ---------------------------------------------------------------------
        # Trim dates prior to first execution date
        # ---------------------------------------------------------------------

        first_rebalance = rebal_dates[0]

        first_rebalance_pos = (
            all_trading_dates
            .get_loc(
                first_rebalance
            )
        )

        first_execution_pos = (
            first_rebalance_pos + 1
        )

        executed_weights = (
            executed_weights
            .iloc[
                first_execution_pos:
            ]
        )

        # ---------------------------------------------------------------------
        # Align asset returns
        # ---------------------------------------------------------------------

        returns_subset = (
            asset_returns
            .reindex(
                index=executed_weights.index,
                columns=executed_weights.columns,
            )
        )

        # ---------------------------------------------------------------------
        # Daily gross portfolio returns
        # ---------------------------------------------------------------------

        portfolio_daily_returns = (
            executed_weights
            * returns_subset
        ).sum(axis=1)

        # ---------------------------------------------------------------------
        # Build result DataFrame
        # ---------------------------------------------------------------------

        result = pd.DataFrame(
            {
                "date": (
                    executed_weights.index
                ),
                "model": model,
                "portfolio": portfolio,
                "rebalancing_frequency": (
                    REBALANCING_FREQUENCY
                ),
                "gross_return": (
                    portfolio_daily_returns.values
                ),
            }
        )

        # ---------------------------------------------------------------------
        # Cumulative gross return
        # ---------------------------------------------------------------------

        result[
            "cumulative_return"
        ] = (
            1.0
            + result["gross_return"]
        ).cumprod() - 1.0

        gross_returns_list.append(
            result
        )

    # -------------------------------------------------------------------------
    # Consolidate Current Frequency
    # -------------------------------------------------------------------------

    frequency_results.append(
        pd.concat(
            gross_returns_list,
            ignore_index=True,
        )
    )


# =============================================================================
# Consolidation — All Frequencies
# =============================================================================

frequency_gross_returns = (
    pd.concat(
        frequency_results,
        ignore_index=True,
    )
    [
        [
            "date",
            "model",
            "portfolio",
            "rebalancing_frequency",
            "gross_return",
            "cumulative_return",
        ]
    ]
    .sort_values(
        [
            "rebalancing_frequency",
            "date",
            "model",
            "portfolio",
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# Execution Audit
# =============================================================================

print("=" * 80)
print(
    "REBALANCING FREQUENCY SENSITIVITY "
    "— EXECUTION AUDIT"
)
print("=" * 80)

print(
    f"✓ Frequencies tested = "
    f"{REBALANCING_FREQUENCIES}"
)

print(
    f"✓ Total observations = "
    f"{len(frequency_gross_returns):,}"
)

print(
    f"✓ Unique frequencies = "
    f"{frequency_gross_returns['rebalancing_frequency'].nunique()}"
)

print(
    f"✓ Unique portfolios = "
    f"{frequency_gross_returns['portfolio'].nunique()}"
)

print(
    f"✓ Unique models = "
    f"{frequency_gross_returns['model'].nunique()}"
)

print(
    f"✓ Date range = "
    f"{frequency_gross_returns['date'].min().date()} "
    f"→ "
    f"{frequency_gross_returns['date'].max().date()}"
)

print(
    f"✓ Missing gross returns = "
    f"{frequency_gross_returns['gross_return'].isna().sum():,}"
)

duplicates = (
    frequency_gross_returns
    .duplicated(
        subset=[
            "date",
            "model",
            "portfolio",
            "rebalancing_frequency",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates:,} "
    "duplicated observations."
)

print(
    f"✓ Duplicate observations = "
    f"{duplicates:,}"
)

print("=" * 80)

print("\nObservations by frequency:")

print(
    frequency_gross_returns[
        "rebalancing_frequency"
    ]
    .value_counts()
    .sort_index()
)

REBALANCING FREQUENCY SENSITIVITY — EXECUTION AUDIT
✓ Frequencies tested = [5, 10, 21, 42, 63]
✓ Total observations = 82,320
✓ Unique frequencies = 5
✓ Unique portfolios = 14
✓ Unique models = 3
✓ Date range = 2025-01-16 → 2026-08-10
✓ Missing gross returns = 0
✓ Duplicate observations = 0

Observations by frequency:
rebalancing_frequency
5     16464
10    16464
21    16464
42    16464
63    16464
Name: count, dtype: int64


### 5.3 Trade-off Analysis: Turnover vs. Net CAGR

In [8]:
# =============================================================================
# Rebalancing Frequency Sensitivity
# =============================================================================

from src.portfolio.utils import calculate_drawdown_metrics
from src.portfolio.utils import calculate_cagr
from src.portfolio.utils import calculate_sharpe
from src.portfolio.utils import calculate_sortino

# =============================================================================
# Configuration
# =============================================================================

REBALANCING_FREQUENCIES = {
    5: "Weekly",
    10: "Biweekly",
    21: "Monthly",
    42: "Bimonthly",
    63: "Quarterly",
}

BASE_TRANSACTION_COST = 0.0015       # 15 bps
TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0

# =============================================================================
# Frequency Sensitivity Engine
# =============================================================================

frequency_results = []
frequency_turnover = []


for frequency, frequency_label in REBALANCING_FREQUENCIES.items():


    # =========================================================================
    # 1. Construct Rebalancing Calendar
    # =========================================================================

    all_weight_dates = (
        weights_raw["date"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    rebalancing_dates = (
        all_weight_dates.iloc[::frequency]
    )


    weights_df = weights_raw[
        weights_raw["date"].isin(
            rebalancing_dates
        )
    ].copy()


    rebal_dates = sorted(
        weights_df["date"].unique()
    )


    # =========================================================================
    # 2. Validate Rebalancing Calendar
    # =========================================================================

    rebal_positions = [
        all_trading_dates.get_loc(date)
        for date in rebal_dates
    ]

    rebal_intervals = np.diff(
        rebal_positions
    )

    assert np.all(
        rebal_intervals == frequency
    ), (
        f"Invalid {frequency}-day "
        "rebalancing calendar."
    )


    # =========================================================================
    # 3. Process Each Portfolio
    # =========================================================================

    grouped_weights = weights_df.groupby(
        ["model", "portfolio"]
    )


    for (model, portfolio), group in grouped_weights:


        # =====================================================================
        # Target Weight Matrix
        # =====================================================================

        weight_matrix = (
            group
            .pivot(
                index="date",
                columns="ticker",
                values="weight",
            )
            .fillna(0.0)
            .reindex(rebalancing_dates)
            .fillna(0.0)
        )


        # =====================================================================
        # Turnover
        # =====================================================================

        previous_weights = (
            weight_matrix.shift(1)
        )

        turnover = (
            0.5
            * (
                weight_matrix
                - previous_weights
            )
            .abs()
            .sum(axis=1)
        )


        # First formation has no previous portfolio
        turnover = turnover.iloc[1:]


        # =====================================================================
        # Map Rebalance Dates to Execution Dates
        # =====================================================================

        execution_dates = []

        for date in turnover.index:

            position = (
                all_trading_dates.get_loc(
                    date
                )
            )

            execution_dates.append(
                all_trading_dates[
                    position + 1
                ]
            )


        turnover_series = pd.Series(
            turnover.values,
            index=execution_dates,
            name="turnover",
        )


        # =====================================================================
        # Executed Buy-and-Hold Weights
        # =====================================================================

        executed_weights = (
            weight_matrix
            .reindex(all_trading_dates)
            .shift(1)
            .ffill()
        )


        # First execution date
        first_rebalance = rebal_dates[0]

        first_rebalance_pos = (
            all_trading_dates.get_loc(
                first_rebalance
            )
        )

        first_execution_pos = (
            first_rebalance_pos + 1
        )

        executed_weights = (
            executed_weights.iloc[
                first_execution_pos:
            ]
        )


        # =====================================================================
        # Align Asset Returns
        # =====================================================================

        returns_subset = (
            asset_returns
            .reindex(
                index=executed_weights.index,
                columns=executed_weights.columns,
            )
            .fillna(0.0)
        )


        # =====================================================================
        # Gross Portfolio Returns
        # =====================================================================

        gross_returns = (
            executed_weights
            * returns_subset
        ).sum(axis=1)


        # =====================================================================
        # Transaction Costs
        # =====================================================================

        transaction_cost = (
            turnover_series
            .reindex(
                gross_returns.index
            )
            .fillna(0.0)
            * BASE_TRANSACTION_COST
        )


        # =====================================================================
        # Net Portfolio Returns
        # =====================================================================

        net_returns = (
            gross_returns
            - transaction_cost
        )


        # =====================================================================
        # Performance Metrics
        # =====================================================================

        cagr = calculate_cagr(
            net_returns
        )

        annualized_volatility = (
            net_returns.std(ddof=1)
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

        sharpe = calculate_sharpe(
            net_returns
        )

        sortino = calculate_sortino(
            net_returns
        )

        (
            maximum_drawdown,
            average_drawdown,
            maximum_underwater_duration_days,
        ) = calculate_drawdown_metrics(
            net_returns
        )


        # =====================================================================
        # Annualized Turnover
        # =====================================================================

        mean_turnover = (
            turnover_series.mean()
        )

        rebalancings_per_year = (
            TRADING_DAYS_PER_YEAR
            / frequency
        )

        annualized_turnover = (
            mean_turnover
            * rebalancings_per_year
        )


        # =====================================================================
        # Cumulative Returns
        # =====================================================================

        cumulative_gross_return = (
            1.0 + gross_returns
        ).prod() - 1.0

        cumulative_net_return = (
            1.0 + net_returns
        ).prod() - 1.0

        cumulative_transaction_cost = (
            transaction_cost.sum()
        )


        # =====================================================================
        # Store Performance Result
        # =====================================================================

        frequency_results.append(
            {
                "frequency_days": frequency,
                "frequency_label": frequency_label,
                "model": model,
                "portfolio": portfolio,

                "CAGR": cagr,

                "annualized_turnover":
                    annualized_turnover,

                "mean_turnover":
                    mean_turnover,

                "cumulative_gross_return":
                    cumulative_gross_return,

                "cumulative_net_return":
                    cumulative_net_return,

                "cumulative_transaction_cost":
                    cumulative_transaction_cost,

                "annualized_volatility":
                    annualized_volatility,

                "Sharpe":
                    sharpe,

                "Sortino":
                    sortino,

                "maximum_drawdown":
                    maximum_drawdown,

                "average_drawdown":
                    average_drawdown,

                "maximum_underwater_duration_days":
                    maximum_underwater_duration_days,
            }
        )


        # =====================================================================
        # Store Turnover Series
        # =====================================================================

        turnover_frequency_df = pd.DataFrame(
            {
                "date":
                    turnover_series.index,

                "frequency_days":
                    frequency,

                "frequency_label":
                    frequency_label,

                "model":
                    model,

                "portfolio":
                    portfolio,

                "turnover":
                    turnover_series.values,
            }
        )

        frequency_turnover.append(
            turnover_frequency_df
        )


# =============================================================================
# Consolidate Results
# =============================================================================

frequency_sensitivity = (
    pd.DataFrame(
        frequency_results
    )
    .sort_values(
        [
            "frequency_days",
            "model",
            "portfolio",
        ]
    )
    .reset_index(drop=True)
)


frequency_turnover = (
    pd.concat(
        frequency_turnover,
        ignore_index=True,
    )
    .sort_values(
        [
            "frequency_days",
            "model",
            "portfolio",
            "date",
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# Audit
# =============================================================================

print()
print("=" * 80)
print("REBALANCING FREQUENCY SENSITIVITY — AUDIT")
print("=" * 80)

print(
    f"✓ Frequencies tested = "
    f"{list(REBALANCING_FREQUENCIES.keys())}"
)

print(
    f"✓ Total strategy-frequency observations = "
    f"{len(frequency_sensitivity):,}"
)

print(
    f"✓ Unique models = "
    f"{frequency_sensitivity['model'].nunique()}"
)

print(
    f"✓ Unique portfolios = "
    f"{frequency_sensitivity['portfolio'].nunique()}"
)

print(
    f"✓ Missing CAGR = "
    f"{frequency_sensitivity['CAGR'].isna().sum():,}"
)

print(
    f"✓ Missing Sharpe = "
    f"{frequency_sensitivity['Sharpe'].isna().sum():,}"
)

print(
    f"✓ Missing turnover = "
    f"{frequency_sensitivity['annualized_turnover'].isna().sum():,}"
)

print()

for frequency in REBALANCING_FREQUENCIES:

    rebalancing_dates = (
        all_weight_dates
        .iloc[::frequency]
    )

    n_rebalances = len(rebalancing_dates)

    print(
        f" {frequency:>2} trading days "
        f"→ {n_rebalances:>3} rebalancings"
    )

print("=" * 80)


REBALANCING FREQUENCY SENSITIVITY — AUDIT
✓ Frequencies tested = [5, 10, 21, 42, 63]
✓ Total strategy-frequency observations = 210
✓ Unique models = 3
✓ Unique portfolios = 14
✓ Missing CAGR = 0
✓ Missing Sharpe = 0
✓ Missing turnover = 0

  5 trading days →  75 rebalancings
 10 trading days →  38 rebalancings
 21 trading days →  18 rebalancings
 42 trading days →   9 rebalancings
 63 trading days →   6 rebalancings


In [9]:
# =============================================================================
# Table 1 — Frequency Sensitivity: Core Economic Metrics
# =============================================================================

frequency_core = (
    frequency_sensitivity
    .groupby(
        ["frequency_days", "frequency_label"]
    )
    .agg(
        mean_CAGR=("CAGR", "mean"),
        mean_annualized_turnover=(
            "annualized_turnover",
            "mean",
        ),
        mean_transaction_cost=(
            "cumulative_transaction_cost",
            "mean",
        ),
        mean_net_return=(
            "cumulative_net_return",
            "mean",
        ),
    )
    .reset_index()
)

print("=" * 80)
print("TABLE 1 — REBALANCING FREQUENCY: ECONOMIC TRADE-OFF")
print("=" * 80)

print(
    frequency_core.to_string(
        index=False,
        formatters={
            "mean_CAGR":
                lambda x: f"{x:.2%}",

            "mean_annualized_turnover":
                lambda x: f"{x:.2%}",

            "mean_transaction_cost":
                lambda x: f"{x:.2%}",

            "mean_net_return":
                lambda x: f"{x:.2%}",
        },
    )
)

TABLE 1 — REBALANCING FREQUENCY: ECONOMIC TRADE-OFF
 frequency_days frequency_label mean_CAGR mean_annualized_turnover mean_transaction_cost mean_net_return
              5          Weekly    32.66%                  925.35%                 2.04%          56.06%
             10        Biweekly    32.36%                  646.73%                 1.42%          55.49%
             21         Monthly    34.89%                  429.11%                 0.91%          60.03%
             42       Bimonthly    34.21%                  239.84%                 0.48%          58.54%
             63       Quarterly    33.89%                  170.43%                 0.32%          58.15%


El examen de los costes de transacción frente al rendimiento bruto generado revela un claro punto de inflexión en la eficiencia financiera del sistema. Al espaciar la ejecución de las órdenes, la rotación anualizada de las carteras (turnover) experimenta un descenso drástico, reduciéndose desde un $925.35\%$ en la frecuencia semanal hasta un $170.43\%$ en la trimestral. Esta contracción drástica del volumen negociado disminuye directamente el impacto de las comisiones y el deslizamiento de precios (slippage), reduciendo el coste transaccional medio del $2.04\%$ al $0.32\%$.

Lejos de penalizar la captura de rentabilidad, la reducción en la frecuencia de rebalanceo optimiza el resultado final. El rendimiento bruto (CAGR) y el retorno neto alcanzan su máximo global en el horizonte mensual ($21$ días), registrando un $34.89\%$ y un $60.03\%$, respectivamente. Las frecuencias de ultra corto plazo ($5$ y $10$ días) erosionan el capital neto mediante la acumulación excesiva de costes operativos sin aportar un alpha adicional significativo. Por su parte, la extensión hacia periodos bimestrales y trimestrales ($42$ y $63$ días) inicia un leve proceso de degradación de la señal predictiva (signal decay), situando los retornos netos promedio en el entorno del $58\%$.

In [10]:
# =============================================================================
# Table 2 — Frequency Sensitivity: Risk-Adjusted Performance
# =============================================================================

frequency_risk = (
    frequency_sensitivity
    .groupby(
        ["frequency_days", "frequency_label"]
    )
    .agg(
        mean_Sharpe=("Sharpe", "mean"),
        mean_Sortino=("Sortino", "mean"),
        mean_MaxDD=("maximum_drawdown", "mean"),
        mean_Underwater_Days=(
            "maximum_underwater_duration_days",
            "mean",
        ),
    )
    .reset_index()
)

print("=" * 80)
print("TABLE 2 — REBALANCING FREQUENCY: RISK-ADJUSTED PERFORMANCE")
print("=" * 80)

print(
    frequency_risk.to_string(
        index=False,
        formatters={
            "mean_Sharpe":
                lambda x: f"{x:.3f}",

            "mean_Sortino":
                lambda x: f"{x:.3f}",

            "mean_MaxDD":
                lambda x: f"{x:.2%}",

            "mean_Underwater_Days":
                lambda x: f"{x:.1f}",
        },
    )
)

TABLE 2 — REBALANCING FREQUENCY: RISK-ADJUSTED PERFORMANCE
 frequency_days frequency_label mean_Sharpe mean_Sortino mean_MaxDD mean_Underwater_Days
              5          Weekly       1.355        2.027    -23.02%                 94.5
             10        Biweekly       1.331        1.995    -23.18%                 97.8
             21         Monthly       1.467        2.209    -22.46%                 92.2
             42       Bimonthly       1.472        2.192    -23.05%                 99.7
             63       Quarterly       1.403        2.089    -23.32%                 91.9


El estudio del comportamiento del riesgo a lo largo de las distintas ventanas de ejecución confirma que el intervalo comprendido entre las tres y las seis semanas maximiza la consistencia de las estrategias. La eficiencia ajustada por riesgo alcanza su cénit en las frecuencias mensual ($21$ días) y bimestral ($42$ días), registrando las lecturas más elevadas del ratio de Sharpe ($1.467$ y $1.472$) y del ratio de Sortino ($2.209$ y $2.192$).

En contraste con las métricas de rentabilidad, las métricas de caída severa muestran un comportamiento prácticamente invariable ante la frecuencia de rebalanceo. El drawdown máximo promedio se mantiene acotado en una franja sumamente estrecha que oscila entre el $-22.46\%$ mensual y el $-23.32\%$ trimestral. Esto demuestra que la profundidad de las caídas de cola es una propiedad inherente a la selección de activos y a la arquitectura del modelo de aprendizaje automático, y no al ritmo de ajuste de las posiciones. Adicionalmente, el tiempo promedio de permanencia en números rojos (underwater duration) encuentra sus mínimos en las ejecuciones trimestral ($91.9$ días) y mensual ($92.2$ días), frente a los $99.7$ días observados en el ajuste bimestral.

In [11]:
# =============================================================================
# Table 3 — Best Strategies Across All Rebalancing Frequencies
# =============================================================================

best_overall = (
    frequency_sensitivity
    .sort_values(
        "CAGR",
        ascending=False,
    )
    .head(20)
)

print("=" * 80)
print("TABLE 3 — TOP 20 STRATEGIES ACROSS ALL REBALANCING FREQUENCIES")
print("=" * 80)

print(
    best_overall[
        [
            "frequency_days",
            "frequency_label",
            "model",
            "portfolio",
            "CAGR",
            "annualized_turnover",
            "Sharpe",
            "Sortino",
            "maximum_drawdown",
        ]
    ].to_string(
        index=False,
        formatters={
            "CAGR":
                lambda x: f"{x:.2%}",

            "annualized_turnover":
                lambda x: f"{x:.2%}",

            "Sharpe":
                lambda x: f"{x:.3f}",

            "Sortino":
                lambda x: f"{x:.3f}",

            "maximum_drawdown":
                lambda x: f"{x:.2%}",
        },
    )
)

TABLE 3 — TOP 20 STRATEGIES ACROSS ALL REBALANCING FREQUENCIES
 frequency_days frequency_label         model                      portfolio   CAGR annualized_turnover Sharpe Sortino maximum_drawdown
             21         Monthly Random Forest long_only_top_10_signal_weight 67.01%             187.73%  1.825   2.691          -32.58%
              5          Weekly Random Forest long_only_top_10_signal_weight 67.00%             378.45%  1.813   2.670          -32.46%
             10        Biweekly Random Forest long_only_top_10_signal_weight 65.06%             258.91%  1.766   2.603          -32.44%
              5          Weekly       XGBoost long_only_top_10_signal_weight 64.30%             438.63%  1.796   2.665          -31.66%
             63       Quarterly Random Forest long_only_top_10_signal_weight 63.16%             104.88%  1.755   2.573          -32.71%
             10        Biweekly       XGBoost long_only_top_10_signal_weight 62.77%             296.71%  1.756   2.605   

El análisis cruzado de las configuraciones individuales más destacadas pone de manifiesto la solidez estructural de las arquitecturas no lineales y la relevancia del control de la rotación. Las diez mejores combinaciones de todo el estudio corresponden unánimemente a la estrategia Long-Only ponderada por intensidad de señal (long_only_top_10_signal_weight), impulsada indistintamente por los algoritmos Random Forest y XGBoost. Este hallazgo confirma que la asignación proporcional a la convicción del modelo es la metodología más alpha-generativa del trabajo, independientemente del horizonte temporal evaluado.

Asimismo, los modelos demuestran una capacidad notable para retener el valor predictivo en horizontes prolongados. La variante de Random Forest genera una rentabilidad bruta idéntica en la ejecución semanal ($67.00\%$) y en la mensual ($67.01\%$). Sin embargo, la ejecución a $21$ días logra este resultado reduciendo la rotación de activos a la mitad ($187.73\%$ frente a $378.45\%$), elevando el ratio de Sharpe hasta $1.825$. Destaca finalmente el comportamiento de ambos modelos no lineales en el rebalanceo trimestral ($63$ días): ambas configuraciones sobrepasan el $61\%$ de CAGR anualizado sosteniendo un turnover inferior al $105\%$, posicionándose como las alternativas operativamente más viables para mandatos de inversión con restricciones severas de liquidez o costes transaccionales elevados.

### 5.4. Full Cross-Sectional Experimentation Grid

Con el fin de ofrecer una visión exhaustiva del espacio de búsqueda, esta sección consolida la totalidad de las iteraciones ejecutadas en el marco experimental. La tabla interactiva expuesta a continuación integra las dimensiones analizadas en las secciones previas: los tres modelos de aprendizaje automático (Ridge, Random Forest y XGBoost), los catorce esquemas de construcción de cartera y las cinco frecuencias de rebalanceo ($5$, $10$, $21$, $42$ y $63$ días).Esta matriz multidimensional permite examinar de manera granular las interacciones entre el algoritmo subyacente, la asignación de pesos y la velocidad de rotación, facilitando la identificación de combinaciones óptimas que maximizan el retorno ajustado por riesgo al tiempo que contienen las fricciones operativas.

In [19]:
import numpy as np
import pandas as pd

# =============================================================================
# Table — Performance & Rebalancing Sensitivity
# =============================================================================

# -------------------------------------------------------------------------
# Define ordering
# -------------------------------------------------------------------------

FREQUENCY_ORDER = {
    5: 0,
    10: 1,
    21: 2,
    42: 3,
    63: 4,
}

MODEL_ORDER = {
    "Ridge": 0,
    "Random Forest": 1,
    "XGBoost": 2,
}


# =============================================================================
# Prepare Table & Calculate Calmar Ratio if not present
# =============================================================================

performance_frequency_table = frequency_sensitivity.copy()

# Calculamos el Calmar Ratio si no existe previamente en el DataFrame
if "Calmar" not in performance_frequency_table.columns:
    performance_frequency_table["Calmar"] = np.where(
        performance_frequency_table["maximum_drawdown"] < 0,
        performance_frequency_table["CAGR"]
        / performance_frequency_table["maximum_drawdown"].abs(),
        np.nan,
    )

# -------------------------------------------------------------------------
# Create temporary sorting keys
# -------------------------------------------------------------------------

performance_frequency_table["_frequency_order"] = (
    performance_frequency_table["frequency_days"].map(FREQUENCY_ORDER)
)

performance_frequency_table["_model_order"] = (
    performance_frequency_table["model"].map(MODEL_ORDER)
)


# =============================================================================
# Sort
# =============================================================================

performance_frequency_table = (
    performance_frequency_table.sort_values(
        [
            "_frequency_order",
            "_model_order",
            "portfolio",
        ]
    )
    .drop(
        columns=[
            "_frequency_order",
            "_model_order",
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# Validation
# =============================================================================

assert set(performance_frequency_table["frequency_days"]) == {
    5,
    10,
    21,
    42,
    63,
}

assert (
    performance_frequency_table[
        [
            "CAGR",
            "annualized_volatility",
            "annualized_turnover",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
        ]
    ]
    .notna()
    .all()
    .all()
)


# =============================================================================
# Audit Output
# =============================================================================

print("=" * 110)
print(
    "PERFORMANCE & REBALANCING SENSITIVITY — "
    "NET RETURNS (BASE SCENARIO — 15 BPS)"
)
print("=" * 110)

print(
    f"✓ Frequencies evaluated = "
    f"{performance_frequency_table['frequency_days'].nunique()}"
)

print(
    f"✓ Total strategy-frequency combinations = "
    f"{len(performance_frequency_table):,}"
)

print()

print(
    performance_frequency_table[
        [
            "frequency_days",
            "frequency_label",
            "model",
            "portfolio",
            "CAGR",
            "annualized_volatility",
            "annualized_turnover",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
        ]
    ].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "annualized_turnover": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)

PERFORMANCE & REBALANCING SENSITIVITY — NET RETURNS (BASE SCENARIO — 15 BPS)
✓ Frequencies evaluated = 5
✓ Total strategy-frequency combinations = 210

 frequency_days frequency_label         model                           portfolio   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
              5          Weekly         Ridge              long_only_equal_weight 45.81%                25.20%             450.88%  1.818   2.767  1.656          -27.66%
              5          Weekly         Ridge long_only_top_10_inverse_volatility 38.43%                23.88%             740.45%  1.609   2.434  1.420          -27.07%
              5          Weekly         Ridge     long_only_top_10_maximum_sharpe 25.52%                22.11%            1549.40%  1.154   1.739  1.085          -23.52%
              5          Weekly         Ridge        long_only_top_10_risk_parity 37.23%                23.02%             831.68%  1.617   2.456  1.465          -25.42

In [25]:
performance_frequency_table.to_parquet(
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet",
    index=False,
)


### 6. Subperiod Performance & Temporal Stability Analysis

Para validar la solidez de los hallazgos no basta con evaluar el rendimiento medio global a lo largo de todo el horizonte *out-of-sample* (OOS). Un retorno acumulado sobresaliente puede ser el resultado engañoso de un único periodo excepcionalmente favorable, ocultando episodios prolongados de mediocridad o volatilidad desmedida. Para responder a una cuestión clave —si la superioridad de una estrategia es estructural o meramente coyuntural—, este apartado introduce una segunda dimensión analítica basada en el **análisis por subperiodos y la consistencia temporal**.

El periodo OOS total se divide en tres fases temporales diferenciadas de similar duración (*Early OOS*, *Middle OOS* y *Late OOS*). Para cada una de estas ventanas y cada combinación de modelo y esquema de ponderación, se replican las métricas de rendimiento neto y comportamiento operativo fundamentales:

$$\text{Métricas por subperiodo} = \{\text{CAGR}, \text{Sharpe Ratio}, \text{Sortino Ratio}, \text{Maximum Drawdown}, \text{Recovery Time}, \text{Turnover}\}$$

**Evaluación de la Consistencia Temporal**

Junto al desglose por fases, se incorpora una métrica cuantitativa directa de persistencia: la **Tasa de Consistencia de Batiendo al Benchmark ($\text{Consistency } \%$)**. Esta variable mide el porcentaje de subperiodos en los que una estrategia dada supera en rendimiento neto ($R^{\text{net}}$) a la referencia pasiva de referencia (*Equal Weight* o referencia del mercado):

$$\text{Consistency } \% = \frac{\sum_{p=1}^{P} \mathbb{I}\left(\text{CAGR}_{\text{Estrategia}, p} > \text{CAGR}_{\text{Benchmark}, p}\right)}{P} \times 100$$

donde $P$ representa el número total de subperiodos evaluados y $\mathbb{I}(\cdot)$ es la función indicadora.

Este enfoque permite discriminar entre arquitecturas que ofrecen alfa genuino y persistente en distintas condiciones de mercado de aquellas dependientes de regímenes específicos, consolidando un marco de comparación estricto para la selección final de modelos.


### 6.1 Subperiod Performance Evaluation (Early, Middle, Late OOS)

In [14]:
import numpy as np
import pandas as pd
from src.portfolio.utils import (
    calculate_cagr,
    calculate_drawdown_metrics,
    calculate_sharpe,
    calculate_sortino,
)


net_returns = pd.read_parquet(
    "../data/portfolio_results/net_portfolio_returns.parquet"
)

# =============================================================================
# Subperiod Performance Analysis
# =============================================================================

TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0
REBALANCING_FREQUENCY = 21


# =============================================================================
# Load & Prepare Net Returns
# =============================================================================

net_returns["date"] = pd.to_datetime(net_returns["date"])

net_returns = net_returns.sort_values(
    ["model", "portfolio", "date"]
).reset_index(drop=True)


# =============================================================================
# Construct Equal-Length OOS Subperiods
# =============================================================================

oos_dates = (
    net_returns["date"].drop_duplicates().sort_values().reset_index(drop=True)
)

n_dates = len(oos_dates)

period_labels = (
    ["Early OOS"] * (n_dates // 3)
    + ["Middle OOS"] * (n_dates // 3)
    + ["Late OOS"] * (n_dates - 2 * (n_dates // 3))
)

date_period_map = pd.DataFrame(
    {
        "date": oos_dates,
        "period": period_labels,
    }
)

net_returns = net_returns.merge(
    date_period_map,
    on="date",
    how="left",
)


# =============================================================================
# Validate Subperiod Construction
# =============================================================================

assert net_returns["period"].notna().all()

period_counts = (
    date_period_map["period"]
    .value_counts()
    .reindex(["Early OOS", "Middle OOS", "Late OOS"])
)

assert period_counts.min() > 0


# =============================================================================
# Load Portfolio Weights for Turnover Calculation
# =============================================================================

weights_raw = pd.read_parquet(
    "../data/portfolio_results/final_portfolio_weights.parquet"
)

weights_raw["date"] = pd.to_datetime(weights_raw["date"])


# =============================================================================
# Rebalancing Calendar
# =============================================================================

rebalancing_dates = (
    weights_raw["date"].drop_duplicates().sort_values().reset_index(drop=True)
)

rebalancing_dates = rebalancing_dates.iloc[::REBALANCING_FREQUENCY]


# =============================================================================
# Calculate Turnover
# =============================================================================

turnover_records = []

grouped_weights = weights_raw.groupby(["model", "portfolio"])

for (model, portfolio), group in grouped_weights:

    weight_matrix = (
        group.pivot(
            index="date",
            columns="ticker",
            values="weight",
        )
        .fillna(0.0)
        .reindex(rebalancing_dates)
        .fillna(0.0)
    )

    previous_weights = weight_matrix.shift(1)

    turnover_series = (
        0.5 * (weight_matrix - previous_weights).abs().sum(axis=1)
    )

    turnover_series = turnover_series.iloc[1:]

    turnover_records.append(
        pd.DataFrame(
            {
                "date": turnover_series.index,
                "model": model,
                "portfolio": portfolio,
                "turnover": turnover_series.values,
            }
        )
    )

turnover_data = pd.concat(turnover_records, ignore_index=True)

# Assign OOS Subperiod
turnover_data = turnover_data.merge(
    date_period_map,
    on="date",
    how="left",
)


# =============================================================================
# Performance Calculation
# =============================================================================

subperiod_results = []

grouped_data = net_returns.groupby(["period", "model", "portfolio"])

for (period, model, portfolio), group in grouped_data:

    group = group.sort_values("date").copy()

    returns = group["net_return_base"].dropna()

    if len(returns) == 0:
        continue

    observations = len(returns)

    # -------------------------------------------------------------------------
    # Performance & Risk Metrics (Fórmulas sin argumentos no admitidos)
    # -------------------------------------------------------------------------

    cagr = calculate_cagr(returns)

    daily_volatility = returns.std(ddof=1)

    annualized_volatility = daily_volatility * np.sqrt(TRADING_DAYS_PER_YEAR)

    sharpe_ratio = calculate_sharpe(returns)

    sortino_ratio = calculate_sortino(returns)

    (
        maximum_drawdown,
        average_drawdown,
        maximum_underwater_duration,
    ) = calculate_drawdown_metrics(returns)

    # -------------------------------------------------------------------------
    # Turnover Calculation
    # -------------------------------------------------------------------------

    turnover_group = turnover_data[
        (turnover_data["model"] == model)
        & (turnover_data["portfolio"] == portfolio)
        & (turnover_data["period"] == period)
    ]

    turnover_series = turnover_group["turnover"].dropna()

    if len(turnover_series) > 0:
        mean_turnover = turnover_series.mean()
        rebalancings_per_year = TRADING_DAYS_PER_YEAR / REBALANCING_FREQUENCY
        annualized_turnover = mean_turnover * rebalancings_per_year
    else:
        mean_turnover = np.nan
        annualized_turnover = np.nan

    # -------------------------------------------------------------------------
    # Store Results
    # -------------------------------------------------------------------------

    subperiod_results.append(
        {
            "period": period,
            "model": model,
            "portfolio": portfolio,
            "observations": observations,
            "CAGR": cagr,
            "annualized_volatility": annualized_volatility,
            "Sharpe": sharpe_ratio,
            "Sortino": sortino_ratio,
            "maximum_drawdown": maximum_drawdown,
            "average_drawdown": average_drawdown,
            "maximum_underwater_duration_days": maximum_underwater_duration,
            "mean_turnover": mean_turnover,
            "annualized_turnover": annualized_turnover,
        }
    )


# =============================================================================
# Consolidate Performance Metrics & Final Order
# =============================================================================

subperiod_performance = (
    pd.DataFrame(subperiod_results)
    .sort_values(
        ["period", "model", "CAGR"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

subperiod_performance = subperiod_performance[
    [
        "period",
        "model",
        "portfolio",
        "observations",
        "CAGR",
        "annualized_volatility",
        "Sharpe",
        "Sortino",
        "maximum_drawdown",
        "average_drawdown",
        "maximum_underwater_duration_days",
        "mean_turnover",
        "annualized_turnover",
    ]
]


# =============================================================================
# Validation
# =============================================================================

expected_periods = {"Early OOS", "Middle OOS", "Late OOS"}

assert set(subperiod_performance["period"]) == expected_periods

assert (
    subperiod_performance[
        [
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "maximum_drawdown",
            "annualized_turnover",
        ]
    ]
    .notna()
    .all()
    .all()
)

assert (
    subperiod_performance["model"].nunique()
    == net_returns["model"].nunique()
)

assert (
    subperiod_performance["portfolio"].nunique()
    == net_returns["portfolio"].nunique()
)

# =============================================================================
# Audit Output
# =============================================================================

print("=" * 80)
print("6.1 — SUBPERIOD PERFORMANCE ANALYSIS")
print("=" * 80)

print(f"✓ Total OOS dates = {len(oos_dates):,}")
print(f"✓ Early OOS dates = {period_counts['Early OOS']:,}")
print(f"✓ Middle OOS dates = {period_counts['Middle OOS']:,}")
print(f"✓ Late OOS dates = {period_counts['Late OOS']:,}")
print(f"✓ Strategies evaluated = {len(subperiod_performance):,}")
print(f"✓ Models evaluated = {subperiod_performance['model'].nunique():,}")
print(
    f"✓ Missing metrics = {subperiod_performance.isna().sum().sum():,}"
)

print()
print("SUBPERIOD DISTRIBUTION")
print("-" * 80)
print(period_counts.to_string())

print()
print("PERIOD DATE RANGES")
print("-" * 80)

for period in ["Early OOS", "Middle OOS", "Late OOS"]:
    period_dates = date_period_map[date_period_map["period"] == period]["date"]
    print(
        f"{period:<12} {period_dates.min().date()} → {period_dates.max().date()}"
    )

print("=" * 80)

6.1 — SUBPERIOD PERFORMANCE ANALYSIS
✓ Total OOS dates = 392
✓ Early OOS dates = 130
✓ Middle OOS dates = 130
✓ Late OOS dates = 132
✓ Strategies evaluated = 126
✓ Models evaluated = 3
✓ Missing metrics = 0

SUBPERIOD DISTRIBUTION
--------------------------------------------------------------------------------
period
Early OOS     130
Middle OOS    130
Late OOS      132

PERIOD DATE RANGES
--------------------------------------------------------------------------------
Early OOS    2025-01-16 → 2025-07-24
Middle OOS   2025-07-25 → 2026-01-29
Late OOS     2026-01-30 → 2026-08-10


### 6.2 Model Stability across Market Regimes

In [15]:
# =============================================================================
# Common Setup & Formatters
# =============================================================================

metrics_to_aggregate = [
    "CAGR",
    "annualized_volatility",
    "Sharpe",
    "Sortino",
    "maximum_drawdown",
    "annualized_turnover",
]

formatters = {
    "CAGR": lambda x: f"{x:.2%}",
    "annualized_volatility": lambda x: f"{x:.2%}",
    "Sharpe": lambda x: f"{x:.3f}",
    "Sortino": lambda x: f"{x:.3f}",
    "maximum_drawdown": lambda x: f"{x:.2%}",
    "annualized_turnover": lambda x: f"{x:.2%}",
}

period_order = ["Early OOS", "Middle OOS", "Late OOS"]

In [16]:
# =============================================================================
# Table 1: Performance Grouped by Model & Subperiod
# =============================================================================

model_summary = (
    subperiod_performance.groupby(["model", "period"])[metrics_to_aggregate]
    .mean()
    .reset_index()
)

# Maintain chronological order of subperiods
model_summary["period"] = pd.Categorical(
    model_summary["period"],
    categories=period_order,
    ordered=True,
)

model_summary = model_summary.sort_values(["model", "period"]).reset_index(
    drop=True
)

print()
print("6.2.A — PERFORMANCE BY MODEL ACROSS SUBPERIODS (AVERAGED)")
print("-" * 80)
print(model_summary.to_string(index=False, formatters=formatters))
print("-" * 80)


6.2.A — PERFORMANCE BY MODEL ACROSS SUBPERIODS (AVERAGED)
--------------------------------------------------------------------------------
        model     period   CAGR annualized_volatility Sharpe Sortino maximum_drawdown annualized_turnover
Random Forest  Early OOS 34.10%                30.29%  1.119   1.693          -22.23%             416.50%
Random Forest Middle OOS 40.27%                18.21%  2.078   3.137           -8.89%             402.44%
Random Forest   Late OOS 35.23%                21.78%  1.608   2.443          -11.39%             462.64%
        Ridge  Early OOS 21.47%                28.38%  0.750   1.115          -22.47%             410.50%
        Ridge Middle OOS 43.39%                17.06%  2.478   3.952           -6.83%             413.69%
        Ridge   Late OOS 30.27%                17.71%  1.661   2.565           -9.57%             454.38%
      XGBoost  Early OOS 33.69%                30.23%  1.119   1.688          -22.63%             437.96%
      XGBoos

El análisis subtemporal muestra un comportamiento altamente consistente en el perfil de riesgo-retorno de las tres arquitecturas, registrando su pico de eficiencia en la fase central OOS. 

XGBoost destaca por ofrecer la trayectoria más equilibrada y superior en el tramo final (Sharpe de 1.740 y CAGR de 37.09%), superando ligeramente a Random Forest (Sharpe 1.608) gracias a una menor volatilidad anualizada (21.00% frente a 21.78%). 

Por su parte, Ridge presenta el perfil más dinámico: aunque sufre una clara compresión en el periodo inicial (Sharpe de 0.750 y maximum drawdown de -22.47%), alcanza el máximo Sharpe global del estudio en el tramo intermedio (2.478 con un drawdown contenido de -6.83%). 

Los niveles de rotación anualizada (turnover) se mantienen elevados y estables en todos los algoritmos (400%–460%), aumentando de forma generalizada en la fase Late OOS.

In [17]:
# =============================================================================
# Table 2: Performance Grouped by Filtered Portfolio Type & Subperiod
# (Includes only Top 10 Long-Only and Long-Short portfolios)
# =============================================================================

# Define filtering patterns (adjust exact string tokens if your naming convention varies)
portfolio_pattern = "top_10|long_short|top10|ls"

# Filter dataset to retain only desired portfolio types
filtered_subperiod = subperiod_performance[
    subperiod_performance["portfolio"].str.contains(
        portfolio_pattern, case=False, regex=True
    )
].copy()

# Aggregate metrics across subperiods
portfolio_summary = (
    filtered_subperiod.groupby(["portfolio", "period"])[metrics_to_aggregate]
    .mean()
    .reset_index()
)

# Maintain chronological order of subperiods
portfolio_summary["period"] = pd.Categorical(
    portfolio_summary["period"],
    categories=period_order,
    ordered=True,
)

portfolio_summary = portfolio_summary.sort_values(
    ["portfolio", "period"]
).reset_index(drop=True)

print()
print("6.2.B — PERFORMANCE BY PORTFOLIO TYPE (TOP 10 & LONG-SHORT) ACROSS SUBPERIODS")
print("-" * 80)
print(portfolio_summary.to_string(index=False, formatters=formatters))
print("-" * 80)


6.2.B — PERFORMANCE BY PORTFOLIO TYPE (TOP 10 & LONG-SHORT) ACROSS SUBPERIODS
--------------------------------------------------------------------------------
                          portfolio     period   CAGR annualized_volatility Sharpe Sortino maximum_drawdown annualized_turnover
long_only_top_10_inverse_volatility  Early OOS 35.29%                35.16%  0.984   1.466          -26.78%             352.05%
long_only_top_10_inverse_volatility Middle OOS 56.46%                21.36%  2.658   4.123           -9.94%             350.97%
long_only_top_10_inverse_volatility   Late OOS 44.48%                23.66%  1.922   2.988          -11.29%             377.76%
    long_only_top_10_maximum_sharpe  Early OOS 27.76%                30.01%  0.927   1.402          -24.52%             671.88%
    long_only_top_10_maximum_sharpe Middle OOS 39.52%                19.26%  2.077   3.216           -9.03%             618.05%
    long_only_top_10_maximum_sharpe   Late OOS 30.89%                21.

Entre las estrategias seleccionadas, la asignación ponderada por señal (long_only_top_10_signal_weight) se erige como la alternativa más generadora de alfa absoluto, alcanzando rendimientos anualizados sobresalientes (hasta un 84.95% de CAGR y Sharpe de 3.540 en Middle OOS) con el turnover más bajo del grupo (~208%–247%). 

No obstante, esta sobreponderación táctica acarrea la mayor volatilidad (30.04% en Late OOS) y caídas máximas más severas (-30.67%). En el extremo opuesto, long_short_equal_weight actúa como un estabilizador del riesgo: limita el drawdown máximo al -4.96% en la fase intermedia y mantiene una volatilidad acotada (~11%–16%), aunque con retornos más modestos. 

Dentro de los esquemas de optimización, maximum_sharpe padece una penalización operativa severa por costes de rotación (rotación anual de hasta el 722.91%), mientras que las aproximaciones risk_parity e inverse_volatility logran un equilibrio óptimo entre captura de retorno y control de caídas.

In [18]:
# =============================================================================
# Table 3: Sharpe Ratio Matrix (Model vs Subperiod Pivot View)
# =============================================================================

sharpe_pivot = subperiod_performance.pivot_table(
    index="model",
    columns="period",
    values="Sharpe",
    aggfunc="mean",
)[period_order]

print()
print("6.2.C — SHARPE RATIO MATRIX (MODEL VS SUBPERIOD)")
print("-" * 80)
print(sharpe_pivot.to_string(float_format=lambda x: f"{x:.3f}"))
print("-" * 80)


6.2.C — SHARPE RATIO MATRIX (MODEL VS SUBPERIOD)
--------------------------------------------------------------------------------
period         Early OOS  Middle OOS  Late OOS
model                                         
Random Forest      1.119       2.078     1.608
Ridge              0.750       2.478     1.661
XGBoost            1.119       2.116     1.740
--------------------------------------------------------------------------------


La matriz de Sharpe confirma la robustez temporal del proceso de aprendizaje, descartando problemas de sobreajuste (overfitting) en la muestra completa. El patrón de comportamiento es uniforme entre modelos: una fase inicial exigente (Early OOS, con promedios de Sharpe entre 0.750 y 1.119), una fuerte expansión de la eficiencia ajustada por riesgo en el periodo central (Middle OOS, superando holgadamente la cota de 2.0 en todas las arquitecturas), y una posterior normalización hacia niveles sostenibles en el tramo final (Late OOS). 

XGBoost demuestra la menor degradación en la transición del tramo medio al tardío (cayendo solo a 1.740), consolidándose como la alternativa más resiliente a los cambios de régimen de mercado frente a Ridge (1.661) y Random Forest (1.608).

Como conclusión  podemos decri que el sistema demuestra que la generación de alfa basada en Machine Learning es sólida y consistente en el tiempo, siendo la combinación de XGBoost con un esquema de ponderación por intensidad de señal (para maximizar retorno) o de paridad de riesgo / long-short (para inmunizar caídas) la arquitectura óptima que equilibra rentabilidad neta, sostenibilidad temporal y costes de ejecución.

## 7. Benchmark Comparison, Alpha Attribution & Rolling Performance

Una vez evaluada la estabilidad temporal del sistema a través de las distintas fases *out-of-sample* (OOS) y analizada la sensibilidad del rendimiento neto ante cambios en la frecuencia de rebalanceo, este capítulo aborda la evaluación económica definitiva del proyecto.

Evaluar una estrategia cuantitativa de forma aislada —por ejemplo, observando únicamente una rentabilidad anualizada absoluta— resulta metodológicamente insuficiente en la gestión de activos moderna. Un rendimiento elevado no implica de forma automática la presencia de habilidad predictiva o superioridad técnica; dicho retorno puede ser el simple reflejo de una exposición pasiva a factores de riesgo sistemático (*Beta* de mercado) o de un comportamiento atípico concentrado en un periodo temporal reducido.

Para validar si el sistema cuantitativo diseñado genera **alfa genuino** (entendido como exceso de rentabilidad ajustado por riesgo y despojado de primas de riesgo tradicionales), este bloque articula un marco de evaluación en cinco fases complementarias:

1. **Formalización de la muestra de carteras candidatas** para enfocar el análisis de atribución en configuraciones representativas del espacio de soluciones.

2. **Descomposición causal del rendimiento** mediante *benchmarking* por componentes, aislando el impacto del algoritmo de *Machine Learning*, la influencia de factores clásicos (*Momentum*) y el aporte del motor de optimización de carteras.

3. **Validación de significación estadística mediante simulación estocástica de Monte Carlo**, contrastando el desempeño del sistema frente a carteras pseudo-aleatorias equivalentes.

4. **Cuantificación de métricas de gestión activa** (*Tracking Error* e *Information Ratio*), evaluando la eficiencia en la conversión de riesgo activo en exceso de retorno.

5. **Diagnóstico de consistencia temporal continua** a través de métricas móviles (*Rolling Performance*), perfiles de *drawdown* históricos y desglose por años naturales.

### 7.1 Candidate Strategy Selection

El proceso de optimización y simulación desarrollado en los capítulos anteriores ha generado un universo amplio de combinaciones resultantes de cruzar arquitecturas de aprendizaje automático (*Ridge*, *Random Forest*, *XGBoost*), esquemas de ponderación (*Signal Weighting*, *Equal Weight*, *Inverse Volatility*, *Risk Parity*, *Maximum Sharpe*, *Long-Short*) y horizontes de selección de activos (*Top 10%*, *Top 20%*, *Top 30%*).

Procesar la totalidad de estas series en las pruebas complejas de atribución de alfa y simulación de Monte Carlo no solo resultaría ineficiente desde el punto de vista computacional, sino que diluiría la claridad ejecutiva del análisis. Por tanto, se establece un filtro transparente y fundamentado para seleccionar **cuatro carteras candidatas representativas** que sintetizan la frontera eficiente del estudio:

* **Estrategia de Máxima Eficiencia (*Highest Sharpe Candidate*):** Representa el punto de óptima conversión de riesgo en retorno en la muestra completa. Esta cartera permite evaluar la capacidad del sistema para maximizar el exceso de rentabilidad por unidad de volatilidad total, minimizando el impacto del *churning* y las fricciones de ejecución.

* **Estrategia de Máximo Retorno Absoluto (*Highest CAGR Candidate*):** Selecciona la configuración que ha liderado la generación bruta y neta de capital a lo largo del periodo *out-of-sample*. Esta alternativa explota la máxima intensidad predictiva del modelo, asumiendo una mayor concentración y volatilidad a cambio de capturar la cola superior de la distribución de retornos (*tail alpha*).

* **Estrategia Defensiva y Control de Riesgo (*Defensive / High Calmar Candidate*):** Escoge la arquitectura enfocada en la preservación del patrimonio (caracterizada por un *Maximum Drawdown* acotado y una rápida velocidad de recuperación). Esta selección evalúa la capacidad de las estrategias neutrales o de baja varianza para inmunizar el capital durante fases de tensión en los mercados financieros.

* **Estrategia Base ML (*Baseline ML Candidate*):** Definida como la combinación de las predicciones del modelo con una asignación pasiva de igual ponderación (*Top 10% Equal Weight*). Esta cartera actúa como el eslabón fundamental de control, permitiendo separar la habilidad puramente predictiva del algoritmo de cualquier sesgo introducido por las técnicas avanzadas de construcción de carteras.

In [43]:
# =============================================================================
# Configuration
# =============================================================================

FREQUENCY_SENSITIVITY_PATH = (
    "../data/portfolio_results/"
    "rebalancing_frequency_sensitivity_metrics.parquet"
)

TOP_N_CANDIDATES = 5


# =============================================================================
# Load Performance Universe
# =============================================================================

performance_frequency_table = pd.read_parquet(
    FREQUENCY_SENSITIVITY_PATH
)

performance_frequency_table["frequency_days"] = (
    performance_frequency_table["frequency_days"]
    .astype(int)
)


# =============================================================================
# Required Columns Validation
# =============================================================================

REQUIRED_COLUMNS = [
    "frequency_days",
    "frequency_label",
    "model",
    "portfolio",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in performance_frequency_table.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)


# =============================================================================
# Candidate Universe Validation
# =============================================================================

assert (
    performance_frequency_table[
        [
            "CAGR",
            "Sharpe",
            "Calmar",
        ]
    ]
    .notna()
    .all()
    .all()
)


# =============================================================================
# Top Candidates by Selection Criterion
# =============================================================================

top_sharpe = (
    performance_frequency_table
    .sort_values(
        "Sharpe",
        ascending=False,
    )
    .head(TOP_N_CANDIDATES)
    .copy()
)

top_cagr = (
    performance_frequency_table
    .sort_values(
        "CAGR",
        ascending=False,
    )
    .head(TOP_N_CANDIDATES)
    .copy()
)

top_calmar = (
    performance_frequency_table
    .sort_values(
        "Calmar",
        ascending=False,
    )
    .head(TOP_N_CANDIDATES)
    .copy()
)


# =============================================================================
# Display Candidate Shortlists
# =============================================================================

DISPLAY_COLUMNS = [
    "frequency_days",
    "model",
    "portfolio",
    "CAGR",
    "annualized_volatility",
    "Sharpe",
    "Sortino",
    "Calmar", 
    "maximum_drawdown",
]


print("=" * 100)
print("TOP 5 — HIGHEST SHARPE CANDIDATES")
print("=" * 100)

print(
    top_sharpe[
        DISPLAY_COLUMNS
    ].to_string(
        index=False,
        formatters={
            "CAGR":
                lambda x: f"{x:.2%}",
            "annualized_volatility":
                lambda x: f"{x:.2%}",
            "annualized_turnover":
                lambda x: f"{x:.2%}",
            "Sharpe":
                lambda x: f"{x:.3f}",
            "Sortino":
                lambda x: f"{x:.3f}",
            "Calmar":
                lambda x: f"{x:.3f}",
            "maximum_drawdown":
                lambda x: f"{x:.2%}",
        },
    )
)


print()
print("=" * 100)
print("TOP 5 — HIGHEST CAGR CANDIDATES")
print("=" * 100)

print(
    top_cagr[
        DISPLAY_COLUMNS
    ].to_string(
        index=False,
        formatters={
            "CAGR":
                lambda x: f"{x:.2%}",
            "annualized_volatility":
                lambda x: f"{x:.2%}",
            "annualized_turnover":
                lambda x: f"{x:.2%}",
            "Sharpe":
                lambda x: f"{x:.3f}",
            "Sortino":
                lambda x: f"{x:.3f}",
            "Calmar":
                lambda x: f"{x:.3f}",
            "maximum_drawdown":
                lambda x: f"{x:.2%}",
        },
    )
)


print()
print("=" * 100)
print("TOP 5 — HIGHEST CALMAR CANDIDATES")
print("=" * 100)

print(
    top_calmar[
        DISPLAY_COLUMNS
    ].to_string(
        index=False,
        formatters={
            "CAGR":
                lambda x: f"{x:.2%}",
            "annualized_volatility":
                lambda x: f"{x:.2%}",
            "annualized_turnover":
                lambda x: f"{x:.2%}",
            "Sharpe":
                lambda x: f"{x:.3f}",
            "Sortino":
                lambda x: f"{x:.3f}",
            "Calmar":
                lambda x: f"{x:.3f}",
            "maximum_drawdown":
                lambda x: f"{x:.2%}",
        },
    )
)

TOP 5 — HIGHEST SHARPE CANDIDATES
 frequency_days model                      portfolio   CAGR annualized_volatility Sharpe Sortino Calmar maximum_drawdown
             21 Ridge long_only_top_10_signal_weight 50.90%                26.25%  1.939   2.937  1.889          -26.94%
              5 Ridge long_only_top_10_signal_weight 49.87%                26.38%  1.890   2.853  1.790          -27.86%
             21 Ridge         long_only_equal_weight 47.60%                25.45%  1.871   2.850  1.793          -26.54%
             42 Ridge long_only_top_10_signal_weight 48.41%                26.12%  1.854   2.780  1.737          -27.87%
             10 Ridge long_only_top_10_signal_weight 49.11%                26.51%  1.852   2.805  1.770          -27.74%

TOP 5 — HIGHEST CAGR CANDIDATES
 frequency_days         model                      portfolio   CAGR annualized_volatility Sharpe Sortino Calmar maximum_drawdown
             21 Random Forest long_only_top_10_signal_weight 67.01%           

In [75]:
# =============================================================================
# 1. Define Selected Candidate Strategies
# =============================================================================

CANDIDATE_KEYS = [
    {
        "role": "Highest Sharpe",
        "model": "Ridge",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest CAGR",
        "model": "Random Forest",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest Calmar",
        "model": "Random Forest",
        "portfolio": "long_only_top_30_maximum_sharpe",
        "frequency_days": 42,
    },
    {
        "role": "Baseline ML",
        "model": "Ridge",
        "portfolio": "long_only_equal_weight",
        "frequency_days": 21,
    },
]

# Merge metrics for audit output
candidates_keys_df = pd.DataFrame(CANDIDATE_KEYS)

selected_candidates_table = candidates_keys_df.merge(
    performance_frequency_table,
    on=["model", "portfolio", "frequency_days"],
    how="inner",
)

# =============================================================================
# 2. Audit Output — 4 Selected Candidates
# =============================================================================

print("=" * 110)
print("SELECTED CANDIDATE STRATEGIES")
print("=" * 110)

print(
    selected_candidates_table[
        [
            "role",
            "model",
            "portfolio",
            "frequency_days",
            "CAGR",
            "annualized_volatility",
            "Sharpe",
            "Sortino",
            "Calmar",
            "maximum_drawdown",
        ]
    ].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)

# =============================================================================
# 3. Extract Target & Executed Weights for Selected Candidates (MultiIndex)
# =============================================================================

candidate_target_dict = {}
candidate_executed_dict = {}

all_weight_dates = (
    weights_raw["date"].drop_duplicates().sort_values().reset_index(drop=True)
)

for config in CANDIDATE_KEYS:
    role = config["role"]
    model = config["model"]
    portfolio = config["portfolio"]
    frequency = config["frequency_days"]

    # Identificador único para el diccionario
    key = (role, model, portfolio, frequency)

    # Rebalancing calendar
    rebalancing_dates = all_weight_dates.iloc[::frequency]

    # Filter raw weights
    mask = (
        (weights_raw["model"] == model)
        & (weights_raw["portfolio"] == portfolio)
        & (weights_raw["date"].isin(rebalancing_dates))
    )
    group = weights_raw[mask].copy()

    # Target Weights Matrix
    weight_matrix = (
        group.pivot(index="date", columns="ticker", values="weight")
        .fillna(0.0)
        .reindex(rebalancing_dates)
        .fillna(0.0)
    )

    # Daily Executed Weights Matrix
    first_rebalance_pos = all_trading_dates.get_loc(rebalancing_dates.iloc[0])
    first_execution_pos = first_rebalance_pos + 1

    executed_weights = (
        weight_matrix.reindex(all_trading_dates)
        .shift(1)
        .ffill()
        .iloc[first_execution_pos:]
    )

    # Guardar en diccionario usando la clave identificadora
    candidate_target_dict[key] = weight_matrix
    candidate_executed_dict[key] = executed_weights

# =============================================================================
# 4. Consolidate into MultiIndex DataFrames & Save
# =============================================================================

index_names = ["role", "model", "portfolio", "frequency_days", "date"]

df_target_weights = pd.concat(
    candidate_target_dict, names=index_names
).fillna(0.0)

df_executed_weights = pd.concat(
    candidate_executed_dict, names=index_names
).fillna(0.0)

print("\n" + "=" * 110)
print("✓ Candidate target & executed weights successfully extracted and saved.")
print("=" * 110)

SELECTED CANDIDATE STRATEGIES
          role         model                       portfolio  frequency_days   CAGR annualized_volatility Sharpe Sortino Calmar maximum_drawdown
Highest Sharpe         Ridge  long_only_top_10_signal_weight              21 50.90%                26.25%  1.939   2.937  1.889          -26.94%
  Highest CAGR Random Forest  long_only_top_10_signal_weight              21 67.01%                36.73%  1.825   2.691  2.057          -32.58%
Highest Calmar Random Forest long_only_top_30_maximum_sharpe              42 29.82%                16.58%  1.799   2.703  2.218          -13.45%
   Baseline ML         Ridge          long_only_equal_weight              21 47.60%                25.45%  1.871   2.850  1.793          -26.54%

✓ Candidate target & executed weights successfully extracted and saved.


En primer lugar se establece la **Estrategia de Máxima Eficiencia (*Highest Sharpe Candidate*)**, materializada a través de la configuración `long_only_top_10_signal_weight` bajo el modelo de regresión regularizada *Ridge* y una frecuencia de rebalanceo mensual de 21 días. Esta alternativa lidera el ranking global de conversión de riesgo en retorno al situarse en el Top 1 absoluto con un Ratio de Sharpe de 1.939, un CAGR del 50.90%, una volatilidad anualizada del 26.25%, un Ratio de Calmar de 1.889 y un Maximum Drawdown del -26.94%. Su selección se justifica al representar el techo analítico de eficiencia total del estudio, demostrando cómo la penalización L2 combinada con una ponderación proporcional a la intensidad de la señal ofrece una relación riesgo-retorno superior dentro del conjunto evaluado.

En segundo lugar se selecciona la **Estrategia de Máximo Retorno Absoluto (*Highest CAGR Candidate*)**, representada por la cartera `long_only_top_10_signal_weight` respaldada por el algoritmo *Random Forest* a 21 días de rebalanceo. Esta arquitectura alcanza el Top 1 en generación neta de capital con un CAGR del 67.01%, acompañado de un Ratio de Sharpe de 1.825, un Ratio de Calmar de 2.057, una volatilidad del 36.73% y una caída máxima del -32.58%. La inclusión de esta variante se fundamenta en su capacidad para explotar las no linealidades de los árboles de decisión y capturar la cola superior de la distribución de rentabilidades, asumiendo un perfil de varianza más elevado que resulta idóneo para estresar las métricas de gestión activa como el Tracking Error y el Information Ratio frente al mercado pasivo.

Como tercer pilar se incorpora la **Estrategia Defensiva y de Control de Riesgo (*Defensive / High Calmar Candidate*)**, configurada mediante la cartera `long_only_top_30_maximum_sharpe` con modelo *Random Forest* y rebalanceo bimestral a 42 días. Esta combinación corona el ranking de conservación patrimonial al obtener el Top 1 de Ratio de Calmar del estudio con un valor de 2.218, comprimiendo el Maximum Drawdown a un -13.45% y la volatilidad a un nivel institucional del 16.58%, sin renunciar a una tasa de crecimiento compuesto del 29.82% anualizado y un Sharpe de 1.799. Su presencia se justifica por la necesidad de evaluar un perfil de baja varianza donde la ampliación del universo al 30% más favorable y la intervención del optimizador de varianza/covarianza reducen la caída máxima a menos de la mitad respecto a las estrategias concentradas.

Finalmente, se formaliza la **Estrategia Base ML (*Baseline ML Candidate*)**, que emplea la estructura `long_only_equal_weight` con el modelo *Ridge* a 21 días. Situada en el Top 3 global de eficiencia con un Ratio de Sharpe de 1.871, un CAGR del 47.60%, una volatilidad del 25.45%, un Calmar de 1.793 y un Maximum Drawdown del -26.54%, esta cartera combina las lecturas predictivas del algoritmo con un esquema no informado de igual ponderación. Su función en la muestra es servir como el elemento de control indispensable dentro del protocolo de desacoplamiento, permitiendo aislar la capacidad discriminatoria intrínseca de la señal de Machine Learning frente a las ganancias de rendimiento atribuibles exclusivamente a las técnicas complejas de optimización de carteras.

Para optimizar la carga computacional en las fases de atribución, las matrices de exposición de las cuatro candidatas se han persistido en disco en formato Parquet a través de dos estructuras complementarias.

Por una parte, los pesos objetivo (target_weights) registran la asignación táctica teórica dictada por el modelo exclusivamente en las fechas de rebalanceo, lo que permite auditar la decisión pura del algoritmo y la rotación de activos. Por otra parte, los pesos ejecutados (executed_weights) proyectan la exposición diaria real de la cartera al aplicar un desfase operativo de un día ($T+1$) tras la señal para eliminar el sesgo de premonición (look-ahead bias), manteniendo las posiciones mediante un esquema buy-and-hold hasta la siguiente ventana. Esta última matriz constituye el insumo principal para el cálculo continuo de retornos, alfas y Tracking Error.

In [ ]:
df_target_weights.to_parquet("../data/portfolio_results/candidate_target_weights.parquet")
df_executed_weights.to_parquet("../data/portfolio_results/candidate_executed_weights.parquet")

### 7.2 Absolute & Factor-Adjusted Benchmarking

Para determinar con rigurosidad científica la procedencia del rendimiento, no basta con demostrar que el sistema completo obtiene plusvalías. Es indispensable responder a tres preguntas metodológicas esenciales: *¿La estrategia supera la exposición pasiva al mercado?*, *¿El rendimiento se debe al algoritmo de Machine Learning o al factor de riesgo subyacente?* y *¿Cuánto valor añade realmente la etapa de optimización de pesos?*

Para aislar el efecto de cada decisión de diseño dentro de la cadena operativa:

$$\text{Predicción ML} \longrightarrow \text{Selección de Señal} \longrightarrow \text{Ponderación de Cartera} \longrightarrow \text{Estrategia Final}$$

se estructuran cuatro *benchmarks* de contraste.


#### 7.2.1 Market Baseline Comparison (Benchmark A)

La primera referencia de evaluación es la estrategia pasiva de comprar y mantener el índice de mercado:

$$\text{Benchmark A} = \text{S\&P 500 Buy \& Hold}$$

Esta comparativa responde a una de las preguntas fundamentales del estudio: ¿genera la gestión activa basada en señales de Machine Learning suficiente valor añadido frente a la mera exposición pasiva al mercado como para justificar la complejidad adicional?

Las carteras candidatas se evalúan frente al índice de referencia utilizando métricas normalizadas de crecimiento patrimonial y comportamiento ajustado por riesgo: Compound Annual Growth Rate (CAGR), Volatilidad Anualizada ($\sigma_{\text{ann}}$), Sharpe Ratio, Sortino Ratio y Maximum Drawdown (MDD). Adicionalmente, se incorpora la Rentabilidad Acumulada (Cumulative Return) para comparar directamente la evolución patrimonial durante todo el periodo out-of-sample.

Para garantizar una comparación económicamente homogénea, las métricas de las estrategias ML se calculan sobre rentabilidades netas de costes de transacción, mientras que el benchmark representa una estrategia buy & hold con una fricción operativa mínima.

El análisis permitirá determinar no solo qué estrategias superan al mercado en términos de rentabilidad absoluta, sino también si dicha superioridad se obtiene mediante una asunción de riesgo significativamente mayor o si las estrategias ML consiguen mejorar simultáneamente la eficiencia riesgo-retorno y el control de pérdidas.


In [66]:
# =============================================================================
# S&P 500 Benchmark — Data Extraction
# =============================================================================

BENCHMARK_TICKER = "^GSPC"

OOS_START_DATE = "2025-01-15"
OOS_END_DATE = "2026-08-11"

benchmark_data = yf.download(
    BENCHMARK_TICKER,
    start=OOS_START_DATE,
    end=OOS_END_DATE,
    auto_adjust=False,
    progress=False,
)

# -----------------------------------------------------------------------------
# Extract Adjusted Close
# -----------------------------------------------------------------------------

benchmark_prices = benchmark_data["Adj Close"]

# yfinance may return a DataFrame even for a single ticker
if isinstance(benchmark_prices, pd.DataFrame):
    benchmark_prices = benchmark_prices.iloc[:, 0]

benchmark_prices = benchmark_prices.copy()

# -----------------------------------------------------------------------------
# Index Formatting
# -----------------------------------------------------------------------------

benchmark_prices.index = pd.to_datetime(
    benchmark_prices.index
)

if benchmark_prices.index.tz is not None:
    benchmark_prices.index = (
        benchmark_prices.index.tz_localize(None)
    )

benchmark_prices.name = "sp500_adj_close"

# =============================================================================
# Audit
# =============================================================================

print("=" * 80)
print("S&P 500 BENCHMARK — DATA EXTRACTION")
print("=" * 80)

print(
    f"✓ Ticker = {BENCHMARK_TICKER}"
)

print(
    f"✓ Date range = "
    f"{benchmark_prices.index.min().date()} "
    f"→ "
    f"{benchmark_prices.index.max().date()}"
)

print(
    f"✓ Observations = "
    f"{len(benchmark_prices):,}"
)

print(
    f"✓ Missing prices = "
    f"{benchmark_prices.isna().sum():,}"
)

print(
    f"✓ Data type = "
    f"{type(benchmark_prices).__name__}"
)

# =============================================================================
# S&P 500 — Daily Returns
# =============================================================================

benchmark_returns = (
    benchmark_prices
    .pct_change()
    .dropna()
    .rename("benchmark_return")
)

print()
print("=" * 80)
print("S&P 500 BENCHMARK — RETURN SERIES")
print("=" * 80)

print(
    f"✓ Return observations = "
    f"{len(benchmark_returns):,}"
)

print(
    f"✓ First return date = "
    f"{benchmark_returns.index.min().date()}"
)

print(
    f"✓ Last return date = "
    f"{benchmark_returns.index.max().date()}"
)

print(
    f"✓ Missing returns = "
    f"{benchmark_returns.isna().sum():,}"
)

S&P 500 BENCHMARK — DATA EXTRACTION
✓ Ticker = ^GSPC
✓ Date range = 2025-01-15 → 2026-08-10
✓ Observations = 390
✓ Missing prices = 0
✓ Data type = Series

S&P 500 BENCHMARK — RETURN SERIES
✓ Return observations = 389
✓ First return date = 2025-01-16
✓ Last return date = 2026-08-10
✓ Missing returns = 0


In [94]:
from src.portfolio.utils import (
    calculate_cagr,
    calculate_drawdown_metrics,
    calculate_sharpe,
    calculate_sortino,
    calculate_benchmark_metrics
)

# =============================================================================
# Global Constants & Configuration
# =============================================================================

TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.0

CANDIDATES = [
    {
        "role": "Highest Sharpe",
        "model": "Ridge",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest CAGR",
        "model": "Random Forest",
        "portfolio": "long_only_top_10_signal_weight",
        "frequency_days": 21,
    },
    {
        "role": "Highest Calmar",
        "model": "Random Forest",
        "portfolio": "long_only_top_30_maximum_sharpe",
        "frequency_days": 42,
    },
    {
        "role": "Baseline ML",
        "model": "Ridge",
        "portfolio": "long_only_equal_weight",
        "frequency_days": 21,
    },
]

METRICS_PATH = (
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)

# =============================================================================
# Main Execution Pipeline
# =============================================================================

# 1. Load precomputed frequency sensitivity metrics
df_all_metrics = pd.read_parquet(METRICS_PATH)

# 2. Extract metrics for selected candidate strategies
candidate_records = []

for candidate in CANDIDATES:
    mask = (
        (df_all_metrics["model"] == candidate["model"])
        & (df_all_metrics["portfolio"] == candidate["portfolio"])
        & (df_all_metrics["frequency_days"] == candidate["frequency_days"])
    )

    row = df_all_metrics.loc[mask].copy()

    if not row.empty:
        row["role"] = candidate["role"]
        candidate_records.append(row)

df_candidates_metrics = pd.concat(candidate_records, ignore_index=True)

# 3. Compute benchmark metrics (expects `benchmark_returns` in current scope)
benchmark_metrics = calculate_benchmark_metrics(benchmark_returns)
benchmark_row = pd.DataFrame([benchmark_metrics])

# 4. Consolidate and apply explicit ordering
market_baseline_comparison = pd.concat(
    [benchmark_row, df_candidates_metrics], ignore_index=True
)

role_order = [
    "Benchmark",
    "Highest Sharpe",
    "Highest CAGR",
    "Highest Calmar",
    "Baseline ML",
]

market_baseline_comparison["role"] = pd.Categorical(
    market_baseline_comparison["role"], categories=role_order, ordered=True
)

market_baseline_comparison = market_baseline_comparison.sort_values(
    "role"
).reset_index(drop=True)

# 5. Formatted audit output
cols_to_show = [
    "role",
    "model",
    "portfolio",
    "frequency_days",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

market_baseline_comparison["frequency_days"] = (
    market_baseline_comparison["frequency_days"]
    .map(lambda x: f"{int(x)}" if pd.notnull(x) else "N/A")
)

print("=" * 110)
print("7.2.1 — MARKET BASELINE COMPARISON")
print("=" * 110)

print(
    market_baseline_comparison[cols_to_show].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "annualized_turnover": lambda x: (
                f"{x:.2%}" if pd.notnull(x) else "N/A"
            ),
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)

7.2.1 — MARKET BASELINE COMPARISON
          role         model                       portfolio frequency_days   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
     Benchmark       S&P 500                    buy_and_hold            N/A 18.71%                17.16%               0.00%  1.090   1.627  0.990          -18.90%
Highest Sharpe         Ridge  long_only_top_10_signal_weight             21 50.90%                26.25%             273.12%  1.939   2.937  1.889          -26.94%
  Highest CAGR Random Forest  long_only_top_10_signal_weight             21 67.01%                36.73%             187.73%  1.825   2.691  2.057          -32.58%
Highest Calmar Random Forest long_only_top_30_maximum_sharpe             42 29.82%                16.58%             423.44%  1.799   2.703  2.218          -13.45%
   Baseline ML         Ridge          long_only_equal_weight             21 47.60%                25.45%             251.29%  1.871   2.850  1.79

Los resultados comparativos confirman que las tres estrategias seleccionadas —Highest Sharpe, Highest CAGR y Highest Calmar— superan con solvencia al S&P 500 (18,71% CAGR; Sharpe 1,090) durante el periodo 2025–2026. Asimismo, logran batir al Baseline ML (47,60% CAGR; Sharpe 1,871), aunque la ventaja en retorno ajustado por riesgo sobre este modelo base resulta estrecha, registrando Ratios de Sharpe muy ajustados entre sí (1,799 a 1,939).

A pesar del excelente desempeño absoluto, destacan dos factores de riesgo a vigilar. Por un lado, la alta volatilidad de las estrategias concentradas genera caídas severas, alcanzando un Maximum Drawdown del -32,58% en la variante de mayor retorno (frente al -18,90% del índice), siendo Highest Calmar la única capaz de contener la pérdida máxima (-13,45%). Por otro lado, la rotación anualizada es notablemente elevada en todas las carteras activas —superando el 423% en el rebalanceo a 42 días—, lo que exige vigilar estrechamente la erosión causada por los costes operativos.


#### 7.2.2 Component-Level Benchmarking & Alpha Attribution

Si una estrategia basada en *Machine Learning* utiliza como variables de entrada indicadores técnicos de tendencia o retornos pasados, existe el riesgo de que el modelo actúe simplemente como un *proxy* complejo de un factor sistemático tradicional, como el **Momentum**. Si comprar directamente el decil con mayor *Momentum* ofrece un resultado equivalente al del modelo ML, el mérito de la rentabilidad debe atribuirse a la prima de riesgo del factor y no a la capacidad analítica del algoritmo.

Para desacoplar el origen del alfa y medir el valor añadido de cada componente del *pipeline*, se definen tres *benchmarks* analíticos:


##### Benchmark B — Momentum Factor + Equal Weight (Evaluación del Algoritmo ML)

Se construye una cartera que selecciona el $10\%$ de los activos con mayor rentabilidad acumulada en la ventana de predicción ($21$ días) y los pondera de forma equitativa:

$$\text{Benchmark B} = \text{Momentum (Top 10\%)} \longrightarrow \text{Equal Weight}$$

Al contrastar la estrategia candidata base $\text{ML (Top 10\%)} \rightarrow \text{Equal Weight}$ frente al $\text{Benchmark B}$, se elimina el efecto de la asignación de pesos y se aísla exclusivamente la capacidad discriminatoria del modelo ML frente a la regla mecánica o simple de *momentum* cross-sectional.

Cabe precisar la decisión de emplear un factor de **Momentum de horizonte corto (21-day Momentum)** en lugar del estándar académico de medio plazo (**12–1 Month Momentum**). Aunque el histórico previo permite construir la señal clásica, evaluar un factor de 12 meses sobre una ventana *out-of-sample* (OOS) de solo ~18 meses (2025–2026) expondría la comparativa a un alto sesgo por el régimen de mercado concreto de ese periodo.

Además, prima la coherencia metodológica: como los modelos de ML predicen la rentabilidad esperada a un horizonte de 21 días, enfrentarlos a inercias de 12 meses desvirtuaría el test de atribución. No se estaría evaluando si el ML selecciona mejor que una regla simple, sino comparando dos horizontes de inversión distintos.

Por último, dada la presencia de fenómenos de reversión a corto plazo (*short-term reversal*) en la muestra, la estrategia se denomina explícitamente **Short-Horizon Momentum (21-day Momentum)** para garantizar un *benchmark* limpio y directamente comparable.


In [41]:
# =============================================================================
# Configuration & Constants
# =============================================================================

MOMENTUM_LOOKBACK = 21
SELECTION_PERCENT = 0.10
MAX_TURNOVER_DESIGN = 0.2
MIN_WEIGHT = 0.005
MAX_WEIGHT = 0.05

TRADING_DAYS_PER_YEAR = 252
REBALANCING_DAYS = 21
REBALANCINGS_PER_YEAR = TRADING_DAYS_PER_YEAR / REBALANCING_DAYS


# =============================================================================
# Load Data & Target Schedule
# =============================================================================

prices = pd.read_parquet("../data/raw/sp500_prices_extended.parquet")
prices.index = pd.to_datetime(prices.index)
adj_close = prices["Adj Close"].copy()

df_target_weights = pd.read_parquet(
    "../data/portfolio_results/candidate_target_weights.parquet"
)

df_temp = df_target_weights.reset_index()

baseline_mask = (df_temp["role"] == "Baseline ML") | (df_temp["model"] == "Ridge")

rebalancing_dates = (
    df_temp.loc[baseline_mask, "date"]
    .drop_duplicates()
    .sort_values()
    .to_list()
)


# =============================================================================
# Compute Signal & Unconstrained Long Weights
# =============================================================================

momentum = adj_close / adj_close.shift(MOMENTUM_LOOKBACK) - 1.0
momentum_rebal = momentum.loc[
    momentum.index.intersection(rebalancing_dates)
].copy()

momentum_rank = momentum_rebal.rank(axis=1, method="first", pct=True)
momentum_selected = momentum_rank >= (1.0 - SELECTION_PERCENT)

raw_weights = momentum_selected.astype(float)
raw_weights = raw_weights.div(raw_weights.sum(axis=1), axis=0).fillna(0.0)

# Format to Long Table for apply_turnover_constraint
df_raw_long = (
    raw_weights.rename_axis("date")
    .reset_index()
    .melt(id_vars="date", var_name="ticker", value_name="weight")
)
df_raw_long["model"] = "21-Day Momentum"
df_raw_long["portfolio"] = "top_10_percent_equal_weight"
df_raw_long = df_raw_long[df_raw_long["weight"] > 0].copy()


# =============================================================================
# Helper: Combined Turnover & Box Constraints
# =============================================================================

from src.portfolio.constraints import apply_turnover_constraint

# =============================================================================
# Execute Pipeline & Audit Diagnosis
# =============================================================================

benchmark_b_constrained_weights = apply_turnover_constraint(
    df_raw_long,
    max_turnover=MAX_TURNOVER_DESIGN,
    min_weight=MIN_WEIGHT,
    max_weight=MAX_WEIGHT,
)

# Compute post-constraint turnover
pivot_weights = benchmark_b_constrained_weights.pivot(
    index="date", columns="ticker", values="weight"
).fillna(0.0)
turnover_series = pivot_weights.diff().abs().sum(axis=1) / 2.0
turnover_series.iloc[0] = pivot_weights.iloc[0].abs().sum() / 2.0

max_turnover_rebal = turnover_series.iloc[1:].max()
mean_turnover_rebal = turnover_series.iloc[1:].mean()
ann_turnover = mean_turnover_rebal * REBALANCINGS_PER_YEAR

print("=" * 80)
print("BENCHMARK B — CONSTRAINED WEIGHTS & TURNOVER DIAGNOSIS")
print("=" * 80)

print(f"✓ Observed Max Turnover :   {max_turnover_rebal:.2%}")
print(f"✓ Observed Annualized Turnover:    {ann_turnover:.2%}")
print(
    f"✓ Min Weight Observed (>= {MIN_WEIGHT:.1%}):    "
    f"{benchmark_b_constrained_weights['weight'].min():.4%}"
)
print(
    f"✓ Max Weight Observed (<= {MAX_WEIGHT:.1%}):    "
    f"{benchmark_b_constrained_weights['weight'].max():.4%}"
)
print("=" * 80)

BENCHMARK B — CONSTRAINED WEIGHTS & TURNOVER DIAGNOSIS
✓ Observed Max Turnover :   28.15%
✓ Observed Annualized Turnover:    210.39%
✓ Min Weight Observed (>= 0.5%):    0.5084%
✓ Max Weight Observed (<= 5.0%):    3.4706%


In [42]:
# =============================================================================
# Transaction Costs Configuration
# =============================================================================

TRANSACTION_COST_BPS_BASE = 15.0
BPS_TO_DECIMAL = 10_000.0

TRADING_DAYS_PER_YEAR = 252
REBALANCING_DAYS = 21
REBALANCINGS_PER_YEAR = TRADING_DAYS_PER_YEAR / REBALANCING_DAYS


# =============================================================================
# Align Schedule & Compute Rebalance-Level Turnover
# =============================================================================

# Load baseline target weights schedule to ensure date alignment
df_target_weights = pd.read_parquet(
    "../data/portfolio_results/candidate_target_weights.parquet"
)

df_temp = df_target_weights.reset_index()
baseline_mask = (df_temp["role"] == "Baseline ML") | (df_temp["model"] == "Ridge")

target_rebalancing_dates = (
    df_temp.loc[baseline_mask, "date"]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

# Filter constrained weights strictly to the aligned rebalancing schedule
df_benchmark_weights = benchmark_b_constrained_weights[
    benchmark_b_constrained_weights["date"].isin(target_rebalancing_dates)
].copy()

# Pivot weights matrix: index = date, columns = ticker
pivot_weights = df_benchmark_weights.pivot(
    index="date", columns="ticker", values="weight"
).fillna(0.0)

# Compute periodic turnover across aligned dates: 0.5 * sum(|w_t - w_{t-1}|)
turnover_series = pivot_weights.diff().abs().sum(axis=1) / 2.0

# For initial portfolio creation at t0, assume full transition from cash
turnover_series.iloc[0] = pivot_weights.iloc[0].abs().sum() / 2.0

df_turnover_schedule = pd.DataFrame(
    {
        "date": pivot_weights.index,
        "turnover": turnover_series.values,
        "transaction_cost_pct": turnover_series.values
        * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL),
        "transaction_cost_bps": turnover_series.values
        * TRANSACTION_COST_BPS_BASE,
    }
)


# =============================================================================
# Aggregate Annualized Cost Impact (Base Scenario: 15 bps)
# =============================================================================

# Exclude initial cash deployment (t0) for steady-state annualized metrics
mean_rebal_turnover = df_turnover_schedule["turnover"].iloc[1:].mean()
ann_turnover = mean_rebal_turnover * REBALANCINGS_PER_YEAR

ann_cost_pct = ann_turnover * (TRANSACTION_COST_BPS_BASE / BPS_TO_DECIMAL)
ann_cost_bps = ann_turnover * TRANSACTION_COST_BPS_BASE


# =============================================================================
# Audit & Output Summary
# =============================================================================

print("=" * 80)
print("BENCHMARK B — TRANSACTION COST IMPACT AUDIT (BASE: 15 BPS)")
print("=" * 80)
print(f"✓ Rebalancing Dates Matched:            {len(target_rebalancing_dates):,}")
print(f"✓ Start Rebalance Date:                 {min(target_rebalancing_dates):%Y-%m-%d}")
print(f"✓ End Rebalance Date:                   {max(target_rebalancing_dates):%Y-%m-%d}")
print("-" * 80)
print(f"✓ Average Turnover per Rebalance:       {mean_rebal_turnover:.2%}")
print(f"✓ Annualized Turnover:                  {ann_turnover:.2%}")
print("-" * 80)
print(f"✓ Annual Transaction Cost (%):          {ann_cost_pct:.4%}")
print(f"✓ Annual Transaction Cost (bps):        {ann_cost_bps:.2f} bps")
print("=" * 80)

BENCHMARK B — TRANSACTION COST IMPACT AUDIT (BASE: 15 BPS)
✓ Rebalancing Dates Matched:            18
✓ Start Rebalance Date:                 2025-01-15
✓ End Rebalance Date:                   2026-06-18
--------------------------------------------------------------------------------
✓ Average Turnover per Rebalance:       17.53%
✓ Annualized Turnover:                  210.39%
--------------------------------------------------------------------------------
✓ Annual Transaction Cost (%):          0.3156%
✓ Annual Transaction Cost (bps):        31.56 bps


In [43]:
# =============================================================================
# 1. Align Asset Daily Returns & Apply 1-Day Execution Lag
# =============================================================================

# Calculate daily asset returns
asset_returns = adj_close.pct_change().fillna(0.0)

# Full evaluation range from first rebalance date to end of price history
full_date_range = asset_returns.loc[
    min(target_rebalancing_dates) : max(adj_close.index)
].index

# Pivot constrained weights (uses constrained & capped weight structure)
pivot_weights = benchmark_b_constrained_weights.pivot(
    index="date", columns="ticker", values="weight"
).fillna(0.0)

# Forward-fill target weights across daily trading schedule
daily_weights = (
    pivot_weights.reindex(full_date_range)
    .ffill()
    .fillna(0.0)
)

# Align returns universe to daily weights matrix
aligned_asset_returns = asset_returns.reindex(
    index=daily_weights.index, columns=daily_weights.columns
).fillna(0.0)

# Shift weights by 1 day to enforce t+1 execution (prevents look-ahead bias)
effective_daily_weights = daily_weights.shift(1).fillna(0.0)

# Compute daily gross portfolio returns
gross_returns = (effective_daily_weights * aligned_asset_returns).sum(axis=1)


# =============================================================================
# 2. Deduct Transaction Costs (15 bps on t+1 Execution Date)
# =============================================================================

# Map transaction costs (% deduction) to decision dates
decision_date_costs = pd.Series(0.0, index=daily_weights.index)
decision_date_costs.loc[df_turnover_schedule["date"]] = df_turnover_schedule[
    "transaction_cost_pct"
].values

# Shift costs by 1 day to align cost deduction with t+1 trade execution
execution_date_costs = decision_date_costs.shift(1).fillna(0.0)

# Compute daily net portfolio returns
net_returns = gross_returns - execution_date_costs


# =============================================================================
# 3. Final Performance Audit
# =============================================================================

cumulative_gross = (1.0 + gross_returns).cumprod() - 1.0
cumulative_net = (1.0 + net_returns).cumprod() - 1.0
total_drag = cumulative_gross.iloc[-1] - cumulative_net.iloc[-1]

print("=" * 80)
print("BENCHMARK B — FINAL CONSTRAINED NET RETURNS AUDIT")
print("=" * 80)
print(f"✓ Weight Constraints Enforced:         Min >= 0.5% | Max <= 5.0%")
print(f"✓ Turnover Cap Enforced:               <= 30.0% per rebalance")
print(f"✓ Base Transaction Cost Applied:       15.0 bps")
print(f"✓ Trade Execution Lag:                 t+1 (No look-ahead bias)")
print("-" * 80)
print(f"✓ Cumulative Gross Return:             {cumulative_gross.iloc[-1]:.2%}")
print(f"✓ Cumulative Net Return:               {cumulative_net.iloc[-1]:.2%}")
print(f"✓ Total Transaction Cost Drag:         {total_drag:.2%}")
print("=" * 80)

BENCHMARK B — FINAL CONSTRAINED NET RETURNS AUDIT
✓ Weight Constraints Enforced:         Min >= 0.5% | Max <= 5.0%
✓ Turnover Cap Enforced:               <= 30.0% per rebalance
✓ Base Transaction Cost Applied:       15.0 bps
✓ Trade Execution Lag:                 t+1 (No look-ahead bias)
--------------------------------------------------------------------------------
✓ Cumulative Gross Return:             103.88%
✓ Cumulative Net Return:               102.82%
✓ Total Transaction Cost Drag:         1.06%


In [51]:

from src.portfolio.utils import (
    calculate_benchmark_metrics,
)

# =============================================================================
# Global Constants & Configuration
# =============================================================================

TRADING_DAYS_PER_YEAR = 252

METRICS_PATH = (
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)

# =============================================================================
# Main Execution Pipeline
# =============================================================================

# 1. Compute standardized metrics for Benchmark B using calculate_benchmark_metrics
benchmark_b_metrics = calculate_benchmark_metrics(
    returns=net_returns,
    trading_days=TRADING_DAYS_PER_YEAR,
)

# Overwrite metadata default fields specifically for Benchmark B
benchmark_b_metrics["role"] = "Benchmark B"
benchmark_b_metrics["model"] = "Rule-Based"
benchmark_b_metrics["portfolio"] = "long_only_equal_weight"
benchmark_b_metrics["frequency_days"] = 21
benchmark_b_metrics["frequency_label"] = "21 days"
benchmark_b_metrics["annualized_turnover"] = ann_turnover

df_benchmark_b = pd.DataFrame([benchmark_b_metrics])

# 2. Load precomputed candidate metrics and filter for Baseline ML (Ridge)
df_all_metrics = pd.read_parquet(METRICS_PATH)

baseline_mask = (
    (df_all_metrics["model"] == "Ridge")
    & (df_all_metrics["portfolio"] == "long_only_equal_weight")
    & (df_all_metrics["frequency_days"] == 21)
)

df_baseline = df_all_metrics.loc[baseline_mask].copy()

if not df_baseline.empty:
    df_baseline["role"] = "Baseline ML"
else:
    raise ValueError("Baseline ML model (Ridge) not found in precomputed metrics.")

# 3. Consolidate Benchmark B vs Baseline ML Comparison
baseline_comparison = pd.concat(
    [df_benchmark_b, df_baseline], ignore_index=True
)

role_order = ["Baseline ML", "Benchmark B"]
baseline_comparison["role"] = pd.Categorical(
    baseline_comparison["role"], categories=role_order, ordered=True
)
baseline_comparison = baseline_comparison.sort_values("role").reset_index(
    drop=True
)


# =============================================================================
# Formatted Audit Output
# =============================================================================

cols_to_show = [
    "role",
    "model",
    "portfolio",
    "frequency_days",
    "CAGR",
    "annualized_volatility",
    "annualized_turnover",
    "Sharpe",
    "Sortino",
    "Calmar",
    "maximum_drawdown",
]

baseline_comparison["frequency_days"] = baseline_comparison[
    "frequency_days"
].map(lambda x: f"{int(x)}" if pd.notnull(x) else "N/A")

print("=" * 110)
print("7.2.2 — BENCHMARK B (MOMENTUM) VS BASELINE ML COMPARISON")
print("=" * 110)

print(
    baseline_comparison[cols_to_show].to_string(
        index=False,
        formatters={
            "CAGR": lambda x: f"{x:.2%}",
            "annualized_volatility": lambda x: f"{x:.2%}",
            "annualized_turnover": lambda x: (
                f"{x:.2%}" if pd.notnull(x) else "N/A"
            ),
            "Sharpe": lambda x: f"{x:.3f}",
            "Sortino": lambda x: f"{x:.3f}",
            "Calmar": lambda x: f"{x:.3f}",
            "maximum_drawdown": lambda x: f"{x:.2%}",
        },
    )
)
print("=" * 110)

7.2.2 — BENCHMARK B (MOMENTUM) VS BASELINE ML COMPARISON
       role      model              portfolio frequency_days   CAGR annualized_volatility annualized_turnover Sharpe Sortino Calmar maximum_drawdown
Baseline ML      Ridge long_only_equal_weight             21 47.60%                25.45%             251.29%  1.871   2.850  1.793          -26.54%
Benchmark B Rule-Based long_only_equal_weight             21 57.37%                29.82%             210.39%  1.924   2.798  2.270          -25.27%


El factor simple de momentum de corto plazo demuestra un desempeño superior en todas las métricas de rentabilidad y riesgo ajustado, alcanzando un 57,37% CAGR (frente al 47,60% del Baseline ML) y un Ratio de Sharpe de 1,924 (frente a 1,871). Asimismo, registra un Maximum Drawdown ligeramente más contenido (-25,27% vs -26,54%), lo que se traduce en un Ratio Calmar superior (2,270 vs 1,793).

Desde el punto de vista operativo, la regla heurística de momentum requiere una rotación anualizada menor (210,39%) que el modelo lineal (251,29%), lo que evidencia una mayor persistencia interperiodo en la selección de activos.

Estos hallazgos confirman que una arquitectura ML lineal simple no logra superar a la inercia del factor de precio de corto plazo por sí sola. Este resultado justifica la necesidad de incorporar esquemas de optimización más sofisticados y modelos no lineales en las estrategias candidatas avanzadas para generar un verdadero alfa exento de la mera exposición al factor momentum.


##### Benchmark C — Momentum Factor + Portfolio Construction (Efecto Cruzado)

Se aplica el esquema de construcción de cartera avanzado (ej. *Risk Parity* o *Maximum Sharpe*) sobre la selección del factor *Momentum* puro:

$$\text{Benchmark C} = \text{Momentum (Top 10\%)} \longrightarrow \text{Optimizer}$$

Esta comparación ($\text{ML + Optimizer}$ vs. $\text{Momentum + Optimizer}$) permite determinar si el modelo de *Machine Learning* genera una señal con mejor perfil de varianza/covarianza para los algoritmos de optimización que la simple ordenación por factores cuantitativos tradicionales.



#### Benchmark D — ML + Equal Weight (Evaluación de la Ponderación de Cartera)

Se fija la selección de activos derivada de la predicción ML y se compara la cartera equiponderada frente a los esquemas sofisticados de asignación de pesos (*Signal Weighting*, *Risk Parity*, *Maximum Sharpe*):

$$\text{Benchmark D} = \text{ML (Top 10\%)} \longrightarrow \text{Equal Weight} \quad \text{vs.} \quad \text{ML (Top 10\%)} \longrightarrow \text{Optimizer}$$

Esta prueba cuantifica exactamente la fracción de retorno ajustado por riesgo que aporta el módulo de construcción de carteras respecto a una asignación pasiva no informada.





### 7.3 Statistical Significance: Random Selection Benchmark (Monte Carlo)

Incluso si una estrategia supera al S&P 500 y a los *benchmarks* factoriales, persiste el riesgo de que la selección de activos por parte del modelo sea el resultado de un acierto aleatorio dentro del espacio de búsqueda o de la captura fortuita de ruido *out-of-sample*.

Para verificar la presencia de una capacidad predictiva genuina de forma estadísticamente rigurosa, se implementa una prueba de hipótesis empírica mediante la **Simulación de Monte Carlo**.



#### 7.3.1 Metodología del Test de Selección Aleatoria

Se simula el comportamiento de $N = 500$ carteras pseudo-aleatorias construidas mediante un proceso estocástico que preserva de forma estricta las restricciones operativas de la estrategia real:

* **Mismo universo y dimensión:** En cada fecha de rebalanceo $t_k$, se seleccionan al azar $K$ activos del universo elegible (donde $K$ coincide exactamente con el número de activos elegidos por el modelo ML, ej. $10\%$ del S&P 500).
* **Misma frecuencia y calendario:** El rebalanceo se ejecuta respetando la misma ventana temporal ($\Delta t = 21$ días) y con el desfase operativo de ejecución ($t_k + 1$).
* **Mismo esquema de ponderación y fricción:** A los activos seleccionados aleatoriamente se les aplica el mismo método de asignación de pesos y se descuenta el coste de transacción base ($15\text{ bps}$).

#### 7.3.2 Construcción de la Distribución Empírica y P-Valor

Para cada simulación $j \in \{1, \dots, N\}$, se calcula la tasa de crecimiento anual compuesta neta, obteniendo la distribución nula de rentabilidades:

$$\left\{ \text{CAGR}_{1}^{\text{random}}, \text{CAGR}_{2}^{\text{random}}, \dots, \text{CAGR}_{N}^{\text{random}} \right\}$$

El **p-valor empírico** de la estrategia ML se define como la proporción de carteras aleatorias que lograron igualar o superar el rendimiento neto de la estrategia candidata ($\text{CAGR}_{\text{ML}}$):

$$p_{\text{empírico}} = \frac{1}{N} \sum_{j=1}^{N} \mathbb{I}\left( \text{CAGR}_{j}^{\text{random}} \ge \text{CAGR}_{\text{ML}} \right)$$

donde $\mathbb{I}(\cdot)$ representa la función indicadora.

Si la estrategia de *Machine Learning* se sitúa en un percentil empírico superior al $95\%$ ($p_{\text{empírico}} < 0.05$), se rechaza la hipótesis nula de selección fortuita, confirmando empíricamente que el modelo extrae un patrón informacional real que supera al azar de forma estadísticamente significativa.

### 7.4 Risk-Adjusted Active Metrics

Una vez demostrada la significación estadística, la evaluación de la gestión activa requiere transformar la comparación de rentabilidades absolutas en una cuantificación del **riesgo activo** asumido para generar dicho exceso de retorno.

Para cada estrategia candidata se calcula la serie temporal de retornos diferenciales diarios respecto al *benchmark* de referencia ($R_{b, t}$):

$$R_{\text{active}, t} = R_{\text{strategy}, t} - R_{b, t}$$

A partir de esta serie de retorno activo, se derivan tres métricas fundamentales de la teoría cuantitativa de carteras:



#### 7.4.1 Excess CAGR

Mide la diferencia directa entre la tasa de crecimiento anual compuesta de la estrategia y la del *benchmark*:

$$\text{Excess CAGR} = \text{CAGR}_{\text{strategy}} - \text{CAGR}_{\text{benchmark}}$$

#### 7.4.2 Tracking Error (TE)

Representa la desviación estándar anualizada de la serie de retornos diferenciales diarios. Cuantifica la volatilidad de las decisiones de desviación que toma la estrategia respecto al índice de referencia:

$$\text{Tracking Error} = \sigma\left( R_{\text{active}} \right) \times \sqrt{252} = \sqrt{\frac{252}{T-1} \sum_{t=1}^{T} \left( R_{\text{active}, t} - \bar{R}_{\text{active}} \right)^2}$$

Un *Tracking Error* elevado indica una cartera marcadamente desalineada de la estructura del *benchmark*, lo que implica la asunción de un riesgo estructural independiente del mercado.

### 7.4.3 Information Ratio (IR)

Es el indicador central de eficiencia en la gestión activa. Mide el exceso de rentabilidad anualizado generado por cada unidad de riesgo activo (*Tracking Error*) asumido:

$$\text{Information Ratio} = \frac{\bar{R}_{\text{strategy, ann}} - \bar{R}_{b, \text{ann}}}{\text{Tracking Error}} = \frac{\left( \bar{R}_{\text{strategy}} - \bar{R}_{b} \right) \times 252}{\sigma\left( R_{\text{active}} \right) \times \sqrt{252}}$$

A diferencia del *Sharpe Ratio* (que penaliza el riesgo total), el *Information Ratio* aísla la habilidad del gestor para remunerar las desviaciones del índice base. Valores de $\text{IR} > 0.50$ se consideran representativos de una estrategia activa sólida, mientras que valores de $\text{IR} > 1.00$ reflejan un nivel excepcional de generación de alfa ajustado por riesgo activo.



### 7.5 Temporal Consistency & Rolling Performance

El cálculo de métricas agregadas sobre un periodo *out-of-sample* completo corre el riesgo de ocultar la inestabilidad temporal de un modelo. Una estrategia puede presentar métricas globales atractivas gracias a un comportamiento extraordinariamente positivo en un único año puntual que compense periodos prolongados de bajo rendimiento o estancamiento.

Para garantizar que la rentabilidad deviene de un alfa consistente y no de eventos aislados, esta sección aplica un análisis continuo de dinamismo temporal.



#### 7.5.1 Rolling 36-Month Sharpe Ratio

Se calcula el *Sharpe Ratio* en ventanas móviles superpuestas de $36$ meses ($756$ sesiones de negociación). Para cada día $t \ge 756$, la métrica se define como:

$$\text{Sharpe}_{36\text{m}, t} = \frac{\bar{R}_{t-756:t} - R_f}{\sigma\left( R_{t-756:t} \right)} \times \sqrt{252}$$

La representación gráfica de esta serie temporal permite evaluar la suavidad en la generación de retorno ajustado por riesgo, detectando fases de degradación del modelo, cambios de régimen de mercado o pérdidas de capacidad predictiva a medida que la muestra evoluciona.



#### 7.5.2 Continuous Drawdown Profile

En lugar de resumir el riesgo de caída en una única cifra estática (*Maximum Drawdown*), se mapea de forma continua la trayectoria de pérdida acumulada desde el máximo histórico anterior ($W_t$ representa el valor patrimonial acumulado en la fecha $t$):

$$DD_t = \frac{W_t}{\max_{s \le t} W_s} - 1, \quad \forall t \in [1, T]$$

Esta serie temporal visibiliza con total precisión la profundidad de las caídas patrimoniales, la frecuencia de las etapas de pérdida y la duración exacta de los periodos de recuperación (*underwater duration*), permitiendo comparar la resiliencia de las carteras candidatas frente al S&P 500 durante episodios reales de tensión en los mercados.



#### 7.5.3 Annual Returns Breakdown (Calendar Year Performance)

Finalmente, se desagrega el rendimiento neto de las carteras candidatas y de los *benchmarks* por años naturales completados en la muestra *out-of-sample*. Esta descomposición en formato tabla y mapa de calor (*heatmap*) verifica la consistencia interanual de la estrategia, confirmando si la tasa de éxito del sistema se mantiene homogénea a lo largo del horizonte de inversión.